In [ ]:
import h5py

file = "/home/shruti/Downloads/3RIMG_01AUG2025_0345_L1C_SGP_V01R00.h5"

with h5py.File(file, 'r') as f:
    print(list(f.keys()))

In [ ]:
def print_structure(name, obj):
    print(name)

with h5py.File(file, 'r') as f:
    f.visititems(print_structure)

In [ ]:
import h5py

file = "/home/shruti/Downloads/3RIMG_01AUG2025_0345_L1C_SGP_V01R00.h5"
with h5py.File(file, 'r') as f:
    
    tir1 = f['IMG_TIR1_TEMP'][:]
    tir2 = f['IMG_TIR2_TEMP'][:]
    mir  = f['IMG_MIR_TEMP'][:]
    wv   = f['IMG_WV_TEMP'][:]

# Function to print stats
def print_stats(name, data):
    print(f"\n{name}")
    print("-"*30)
    print("Shape :", data.shape)
    print("Min   :", np.nanmin(data))
    print("Max   :", np.nanmax(data))
    print("Mean  :", np.nanmean(data))
    print("NaNs  :", np.isnan(data).sum())

# Print all channels
print_stats("TIR1", tir1)
print_stats("TIR2", tir2)
print_stats("MIR", mir)
print_stats("WV", wv)

In [ ]:
with h5py.File(file, 'r') as f:
    print(f['IMG_TIR1_TEMP'].shape)

In [ ]:
with h5py.File(file, 'r') as f:
    for key in f.keys():
        try:
            print(key, f[key].shape)
        except:
            print(key, "No shape")

In [ ]:
with h5py.File(file, 'r') as f:
    tir1 = f['IMG_TIR1'][0]   # remove first dimension
    tir2 = f['IMG_TIR2'][0]
    mir  = f['IMG_MIR'][0]
    wv   = f['IMG_WV'][0]

print("Shape:", tir1.shape)

In [ ]:
print("TIR1 min/max:", tir1.min(), tir1.max())

In [ ]:
with h5py.File(file, 'r') as f:
    tir1 = f['IMG_TIR1'][0]
    tir2 = f['IMG_TIR2'][0]
    mir  = f['IMG_MIR'][0]
    wv   = f['IMG_WV'][0]

    tir1_lut = f['IMG_TIR1_TEMP'][:]
    tir2_lut = f['IMG_TIR2_TEMP'][:]
    mir_lut  = f['IMG_MIR_TEMP'][:]
    wv_lut   = f['IMG_WV_TEMP'][:]

In [ ]:
bt = np.stack([
    tir1_lut[tir1.astype(int)],
    tir2_lut[tir2.astype(int)],
    wv_lut[wv.astype(int)],
    mir_lut[mir.astype(int)]
], axis=-1)

In [ ]:
print(bt.min(), bt.max())

In [ ]:
import glob, os

INSAT_DIR = "/home/shruti/Downloads/insat"


# ── Option A: grab ALL INSAT L1C files in the folder ─────────
pattern = os.path.join(INSAT_DIR, "3RIMG_*_L1C_*.h5")
files   = sorted(glob.glob(pattern))

# ── Option B: if you want only a specific date, set it here ──
# TARGET_DATE = "02AUG2025"
# pattern = os.path.join(INSAT_DIR, f"3RIMG_{TARGET_DATE}_*_L1C_*.h5")
# files   = sorted(glob.glob(pattern))

print(f"Found {len(files)} INSAT files:")
for f in files:
    print(" ", os.path.basename(f))

assert len(files) > 0, (
    f"No files found in {INSAT_DIR}\n"
    f"Pattern used: {pattern}\n"
    "→ Run the diagnostic cell above to check your actual filenames."
)

###### ============================================================
# 0. IMPORTS
# ============================================================
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tensorflow.keras.models import load_model
from pyproj import Proj, Transformer
from PIL import Image
from IPython.display import Image as IPImage, display
import glob
import os
import re
from datetime import datetime
import imageio.v2 as imageio
from tqdm import tqdm

# ============================================================
# 1. CONFIG  ← only section you ever need to edit
# ============================================================
MODEL_PATH   = "rain_no_rain.keras"
INSAT_DIR    = "/home/shruti/Downloads/insat"       # folder with .h5 files
OUTPUT_DIR   = "/home/shruti/Downloads/insat_gif"   # ← your output folder path
FRAMES_DIR   = "/tmp/insat_frames/"
INDIA_EXTENT = [68, 97, 7, 37]
PATCH_SIZE   = 128
STRIDE       = 64
FPS          = 2

os.makedirs(FRAMES_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)   # create output folder if needed

# ============================================================
# 2. HELPERS  (all defined before anything calls them)
# ============================================================
def parse_timestamp(fpath):
    """Extract datetime from filename e.g. 3RIMG_01AUG2025_0015_..."""
    fname = os.path.basename(fpath)
    m = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{4})', fname)
    if m:
        return datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")
    return None


def process_file(fpath):
    """Read one INSAT HDF5 file → (rain_prob_map, tir1_bt, X, Y)"""
    with h5py.File(fpath, 'r') as f:
        tir1     = f['IMG_TIR1'][0]
        tir2     = f['IMG_TIR2'][0]
        mir      = f['IMG_MIR'][0]
        wv       = f['IMG_WV'][0]
        tir1_lut = f['IMG_TIR1_TEMP'][:]
        tir2_lut = f['IMG_TIR2_TEMP'][:]
        mir_lut  = f['IMG_MIR_TEMP'][:]
        wv_lut   = f['IMG_WV_TEMP'][:]
        X        = f['X'][:]
        Y        = f['Y'][:]

    bt_raw = np.stack([
        tir1_lut[tir1.astype(int)],
        tir2_lut[tir2.astype(int)],
        wv_lut[wv.astype(int)],
        mir_lut[mir.astype(int)]
    ], axis=0)                      # (4, H, W)  — order matches training

    tir1_bt = bt_raw[0].copy()     # raw BT saved for visualization

    bt_norm = np.clip(bt_raw, 180, 330)
    bt_norm = (bt_norm - 180) / (330 - 180)
    bt_norm = np.transpose(bt_norm, (1, 2, 0))   # → (H, W, 4)

    H, W, _ = bt_norm.shape
    rain_prob_map = np.zeros((H, W))
    count_map     = np.zeros((H, W))

    for i in range(0, H - PATCH_SIZE + 1, STRIDE):
        for j in range(0, W - PATCH_SIZE + 1, STRIDE):
            patch = np.expand_dims(
                bt_norm[i:i+PATCH_SIZE, j:j+PATCH_SIZE], 0)
            pred  = model.predict(patch, verbose=0)
            rain_prob_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += pred[0, :, :, 1]
            count_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE]     += 1

    rain_prob_map /= (count_map + 1e-6)
    return rain_prob_map, tir1_bt, X, Y


# lon/lat grid cache — computed once, reused for all files
_geo_cache = {}

def get_lonlat(X, Y):
    key = X.tobytes()   # stable content-based cache key
    if key not in _geo_cache:
        xx, yy      = np.meshgrid(X, Y)
        proj_geo    = Proj(proj='geos', h=35786023, lon_0=82.5, sweep='x')
        transformer = Transformer.from_proj(
            proj_geo, "epsg:4326", always_xy=True)
        lon, lat    = transformer.transform(xx, yy)
        _geo_cache[key] = (lon, lat)
        print("  (lon/lat grid computed and cached)")
    return _geo_cache[key]


# ============================================================
# 3. COLORMAPS  (defined once, shared across all frames)
# ============================================================
insat_cmap = ListedColormap(['#B0B0B0', '#00C040'])   # gray=No Rain, green=Rain
bounds     = [-0.5, 0.5, 1.5]
norm       = BoundaryNorm(bounds, insat_cmap.N)

legend_patches = [
    mpatches.Patch(color='#00C040', label='Rain'),
    mpatches.Patch(color='#B0B0B0', label='No Rain'),
]

# ============================================================
# 4. RENDER ONE FRAME
# ============================================================
def render_frame(rain_prob_map, tir1_bt, lon, lat,
                 timestamp, frame_idx, total_frames):

    valid     = np.isfinite(lat) & np.isfinite(lon)
    lon_clean = np.where(valid, lon, 0.0)   # pcolormesh needs finite coords
    lat_clean = np.where(valid, lat, 0.0)

    # Build category map
    category_map = np.full(rain_prob_map.shape, np.nan)
    category_map[valid & (rain_prob_map <= 0.5)] = 0   # No Rain
    category_map[valid & (rain_prob_map >  0.5)] = 1   # Rain

    # Masked arrays → invalid pixels render transparent
    tir1_plot = np.ma.array(tir1_bt,      mask=~valid)
    cat_plot  = np.ma.array(category_map, mask=~valid)

    time_str = timestamp.strftime("%d %b %Y  %H:%M UTC")

    # ── Figure: two panels side-by-side ──────────────────────
    fig, axes = plt.subplots(
        1, 2, figsize=(18, 8),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    fig.patch.set_facecolor('white')

    # ── LEFT PANEL: TIR1 Brightness Temperature ──────────────
    ax1 = axes[0]
    im  = ax1.pcolormesh(
        lon_clean, lat_clean, tir1_plot,
        cmap='RdYlBu_r', vmin=180, vmax=320,
        transform=ccrs.PlateCarree(), shading='auto'
    )
    ax1.coastlines(resolution='10m', linewidth=0.8, color='black')
    ax1.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor='black')
    ax1.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor='dimgray')
    gl1 = ax1.gridlines(draw_labels=True, linewidth=0.4,
                         color='gray', alpha=0.6, linestyle='--')
    gl1.top_labels   = False
    gl1.right_labels = False
    ax1.set_extent(INDIA_EXTENT)
    cb = plt.colorbar(im, ax=ax1, orientation='vertical',
                      pad=0.02, shrink=0.92)
    cb.set_label('Brightness Temperature (K)', fontsize=10)
    ax1.set_title(f"TIR1 Brightness Temperature\n{time_str}",
                  fontsize=12, fontweight='bold')

    # ── RIGHT PANEL: Rain / No-Rain Prediction ────────────────
    ax2 = axes[1]
    ax2.pcolormesh(
        lon_clean, lat_clean, cat_plot,
        cmap=insat_cmap, norm=norm,
        transform=ccrs.PlateCarree(), shading='auto'
    )
    ax2.coastlines(resolution='10m', linewidth=0.8, color='black')
    ax2.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor='black')
    ax2.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor='dimgray')
    gl2 = ax2.gridlines(draw_labels=True, linewidth=0.4,
                         color='gray', alpha=0.6, linestyle='--')
    gl2.top_labels   = False
    gl2.right_labels = False
    ax2.set_extent(INDIA_EXTENT)
    ax2.legend(handles=legend_patches, loc='lower right',
               fontsize=9, framealpha=0.85)
    ax2.set_title(f"INSAT Rain / No-Rain Prediction\n{time_str}",
                  fontsize=12, fontweight='bold')
    # ← rain coverage annotation removed as requested

    # ASCII progress bar in suptitle
    progress = "█" * (frame_idx + 1) + "░" * (total_frames - frame_idx - 1)
    fig.suptitle(
        f"INSAT-3DR  |  Rain Detection          "
        f"[{frame_idx+1:02d}/{total_frames:02d}]  {progress}",
        fontsize=13, fontweight='bold', y=1.005
    )

    plt.tight_layout()
    frame_path = os.path.join(FRAMES_DIR, f"frame_{frame_idx:03d}.png")
    plt.savefig(frame_path, dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return frame_path

# ============================================================
# 5. LOAD MODEL
# ============================================================
model = load_model(MODEL_PATH, compile=False)
print("✅ Model loaded.")

# ============================================================
# 6. DISCOVER + SORT FILES
# ============================================================
pattern = os.path.join(INSAT_DIR, "3RIMG_*_L1C_SGP_V01R00.h5")
files   = sorted(glob.glob(pattern))

print(f"Found {len(files)} files:")
for f in files:
    print(" ", os.path.basename(f))

assert len(files) > 0, (
    f"No files found!\nDirectory : {INSAT_DIR}\nPattern   : {pattern}"
)

# Auto-detect date from first filename
first_ts   = parse_timestamp(files[0])
DATE_LABEL = first_ts.strftime("%d%b%Y").upper()   # e.g. "01AUG2025"

OUTPUT_GIF    = os.path.join(OUTPUT_DIR, f"rain_evolution_{DATE_LABEL}.gif")
OUTPUT_GIF_HQ = os.path.join(OUTPUT_DIR, f"rain_evolution_{DATE_LABEL}_hq.gif")

print(f"Date detected : {DATE_LABEL}")
print(f"Output folder : {OUTPUT_DIR}")
print(f"Output GIF    : {OUTPUT_GIF}")
print(f"Output GIF HQ : {OUTPUT_GIF_HQ}")

# ============================================================
# 7. MAIN LOOP — process all files, render frames
# ============================================================
frame_paths = []

for idx, fpath in enumerate(tqdm(files, desc="Processing INSAT files")):
    timestamp = parse_timestamp(fpath)
    if timestamp is None:
        print(f"  ⚠️  Skipping (can't parse timestamp): {fpath}")
        continue

    print(f"  [{idx+1:02d}/{len(files)}]  {timestamp.strftime('%H:%M UTC')}"
          f"  — {os.path.basename(fpath)}")

    rain_prob_map, tir1_bt, X, Y = process_file(fpath)
    lon, lat = get_lonlat(X, Y)

    fp = render_frame(rain_prob_map, tir1_bt, lon, lat,
                      timestamp, idx, len(files))
    frame_paths.append(fp)

print(f"\n✅ {len(frame_paths)} frames rendered → {FRAMES_DIR}")

# ============================================================
# 8. PREVIEW ALL FRAMES INLINE BEFORE SAVING
# ============================================================
print("Previewing frames...")

n      = len(frame_paths)
ncols  = 3
nrows  = (n + ncols - 1) // ncols   # auto row count for any number of files

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 6))
axes = axes.ravel()

for i, fp in enumerate(frame_paths):
    ts  = parse_timestamp(files[i])
    img = plt.imread(fp)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(
        f"Frame {i+1}  |  {ts.strftime('%H:%M UTC')}",
        fontsize=9, fontweight='bold'
    )

# hide unused axes if frames < grid slots
for j in range(n, len(axes)):
    axes[j].axis('off')

plt.suptitle(f"Frame Preview — {DATE_LABEL}  ({n} frames)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================
# 9. COMPILE → STANDARD GIF  (imageio)
# ============================================================
print("Compiling standard GIF...")

frames    = [imageio.imread(fp) for fp in frame_paths]
durations = [1.0 / FPS] * len(frames)
durations[0]  = 2.5
durations[-1] = 4.0

imageio.mimwrite(
    OUTPUT_GIF,
    frames,
    format='GIF',
    duration=durations,
    loop=0
)
print(f"✅ GIF saved  →  {OUTPUT_GIF}")
print(f"   Frames : {len(frames)}  |  FPS : {FPS}  |  "
      f"Duration : ~{len(frames)/FPS:.0f}s")

# ============================================================
# 10. COMPILE → HQ GIF  (Pillow — optimized, smaller file)
# ============================================================
print("Saving optimized HQ GIF with Pillow...")

pil_frames   = [Image.fromarray(imageio.imread(fp)) for fp in frame_paths]
durations_ms = [int(1000 / FPS)] * len(pil_frames)
durations_ms[0]  = 2500
durations_ms[-1] = 4000

pil_frames[0].save(
    OUTPUT_GIF_HQ,
    save_all=True,
    append_images=pil_frames[1:],
    duration=durations_ms,
    loop=0,
    optimize=True
)
print(f"✅ HQ GIF saved  →  {OUTPUT_GIF_HQ}")

# ============================================================
# 11. DISPLAY BOTH GIFS INLINE IN JUPYTER
# ============================================================
print("\n📽️  Standard GIF:")
display(IPImage(filename=OUTPUT_GIF))

print("📽️  HQ GIF:")
display(IPImage(filename=OUTPUT_GIF_HQ))

In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import tensorflow as tf
from tensorflow.keras.models import load_model
from pyproj import Proj, Transformer
from PIL import Image
from IPython.display import Image as IPImage, display
import glob
import os
import re
from datetime import datetime
import imageio.v2 as imageio
from tqdm import tqdm

# ============================================================
# 1. GPU SETUP
# ============================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        # Allows TF to allocate only as much GPU memory as needed
        # instead of grabbing all VRAM upfront → avoids OOM
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU enabled: {[g.name for g in gpus]}")
else:
    print("⚠️  No GPU found — running on CPU")

# ============================================================
# 2. CONFIG
# ============================================================
MODEL_PATH   = "rain_no_rain.keras"
INSAT_DIR    = "/home/shruti/Downloads/insat/insat_aug_01"
OUTPUT_DIR   = "/home/shruti/Downloads/insat_gif"
FRAMES_DIR   = "/tmp/insat_frames/"
INDIA_EXTENT = [68, 97, 7, 37]
PATCH_SIZE   = 128
STRIDE       = 64
FPS          = 2
BATCH_SIZE   = 64    # ← tune this: increase if GPU has more VRAM (try 128/256)
                     #   decrease if you get OOM errors (try 32/16)

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)


# ============================================================
# 2. HELPERS  (all defined before anything calls them)
# ============================================================
def parse_timestamp(fpath):
    """Extract datetime from filename e.g. 3RIMG_01AUG2025_0015_..."""
    fname = os.path.basename(fpath)
    m = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{4})', fname)
    if m:
        return datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")
    return None


def process_file(fpath):
    """Read one INSAT HDF5 file → (rain_prob_map, tir1_bt, X, Y)
       Uses batched GPU inference — much faster than one patch at a time."""
    with h5py.File(fpath, 'r') as f:
        tir1     = f['IMG_TIR1'][0]
        tir2     = f['IMG_TIR2'][0]
        mir      = f['IMG_MIR'][0]
        wv       = f['IMG_WV'][0]
        tir1_lut = f['IMG_TIR1_TEMP'][:]
        tir2_lut = f['IMG_TIR2_TEMP'][:]
        mir_lut  = f['IMG_MIR_TEMP'][:]
        wv_lut   = f['IMG_WV_TEMP'][:]
        X        = f['X'][:]
        Y        = f['Y'][:]

    bt_raw = np.stack([
        tir1_lut[tir1.astype(int)],
        tir2_lut[tir2.astype(int)],
        wv_lut[wv.astype(int)],
        mir_lut[mir.astype(int)]
    ], axis=0)

    tir1_bt = bt_raw[0].copy()

    bt_norm = np.clip(bt_raw, 180, 330)
    bt_norm = (bt_norm - 180) / (330 - 180)
    bt_norm = np.transpose(bt_norm, (1, 2, 0))   # → (H, W, 4)

    H, W, _ = bt_norm.shape
    rain_prob_map = np.zeros((H, W))
    count_map     = np.zeros((H, W))

    # ── Step 1: collect ALL patch positions ──────────────────
    positions = [
        (i, j)
        for i in range(0, H - PATCH_SIZE + 1, STRIDE)
        for j in range(0, W - PATCH_SIZE + 1, STRIDE)
    ]

    # ── Step 2: extract ALL patches into one array ───────────
    patches = np.stack([
        bt_norm[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        for i, j in positions
    ], axis=0).astype(np.float32)   # shape: (N_patches, 128, 128, 4)

    # ── Step 3: batched GPU prediction ───────────────────────
    # model.predict with batch_size sends chunks to GPU efficiently
    preds = model.predict(
        patches,
        batch_size=BATCH_SIZE,
        verbose=0
    )   # shape: (N_patches, 128, 128, 2)

    # ── Step 4: accumulate results back into the map ─────────
    for (i, j), pred in zip(positions, preds):
        rain_prob_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += pred[:, :, 1]
        count_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE]     += 1

    rain_prob_map /= (count_map + 1e-6)
    return rain_prob_map, tir1_bt, X, Y


# lon/lat grid cache — computed once, reused for all files
_geo_cache = {}

def get_lonlat(X, Y):
    key = X.tobytes()   # stable content-based cache key
    if key not in _geo_cache:
        xx, yy      = np.meshgrid(X, Y)
        proj_geo    = Proj(proj='geos', h=35786023, lon_0=82.5, sweep='x')
        transformer = Transformer.from_proj(
            proj_geo, "epsg:4326", always_xy=True)
        lon, lat    = transformer.transform(xx, yy)
        _geo_cache[key] = (lon, lat)
        print("  (lon/lat grid computed and cached)")
    return _geo_cache[key]


# ============================================================
# 3. COLORMAPS  (defined once, shared across all frames)
# ============================================================
insat_cmap = ListedColormap(['#B0B0B0', '#00C040'])   # gray=No Rain, green=Rain
bounds     = [-0.5, 0.5, 1.5]
norm       = BoundaryNorm(bounds, insat_cmap.N)

legend_patches = [
    mpatches.Patch(color='#00C040', label='Rain'),
    mpatches.Patch(color='#B0B0B0', label='No Rain'),
]

# ============================================================
# 4. RENDER ONE FRAME
# ============================================================
def render_frame(rain_prob_map, tir1_bt, lon, lat,
                 timestamp, frame_idx, total_frames):

    valid     = np.isfinite(lat) & np.isfinite(lon)
    lon_clean = np.where(valid, lon, 0.0)   # pcolormesh needs finite coords
    lat_clean = np.where(valid, lat, 0.0)

    # Build category map
    category_map = np.full(rain_prob_map.shape, np.nan)
    category_map[valid & (rain_prob_map <= 0.5)] = 0   # No Rain
    category_map[valid & (rain_prob_map >  0.5)] = 1   # Rain

    # Masked arrays → invalid pixels render transparent
    tir1_plot = np.ma.array(tir1_bt,      mask=~valid)
    cat_plot  = np.ma.array(category_map, mask=~valid)

    time_str = timestamp.strftime("%d %b %Y  %H:%M UTC")

    # ── Figure: two panels side-by-side ──────────────────────
    fig, axes = plt.subplots(
        1, 2, figsize=(18, 8),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    fig.patch.set_facecolor('white')

    # ── LEFT PANEL: TIR1 Brightness Temperature ──────────────
    ax1 = axes[0]
    im  = ax1.pcolormesh(
        lon_clean, lat_clean, tir1_plot,
        cmap='RdYlBu_r', vmin=180, vmax=320,
        transform=ccrs.PlateCarree(), shading='auto'
    )
    ax1.coastlines(resolution='10m', linewidth=0.8, color='black')
    ax1.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor='black')
    ax1.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor='dimgray')
    gl1 = ax1.gridlines(draw_labels=True, linewidth=0.4,
                         color='gray', alpha=0.6, linestyle='--')
    gl1.top_labels   = False
    gl1.right_labels = False
    ax1.set_extent(INDIA_EXTENT)
    cb = plt.colorbar(im, ax=ax1, orientation='vertical',
                      pad=0.02, shrink=0.92)
    cb.set_label('Brightness Temperature (K)', fontsize=10)
    ax1.set_title(f"TIR1 Brightness Temperature\n{time_str}",
                  fontsize=12, fontweight='bold')

    # ── RIGHT PANEL: Rain / No-Rain Prediction ────────────────
    ax2 = axes[1]
    ax2.pcolormesh(
        lon_clean, lat_clean, cat_plot,
        cmap=insat_cmap, norm=norm,
        transform=ccrs.PlateCarree(), shading='auto'
    )
    ax2.coastlines(resolution='10m', linewidth=0.8, color='black')
    ax2.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor='black')
    ax2.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor='dimgray')
    gl2 = ax2.gridlines(draw_labels=True, linewidth=0.4,
                         color='gray', alpha=0.6, linestyle='--')
    gl2.top_labels   = False
    gl2.right_labels = False
    ax2.set_extent(INDIA_EXTENT)
    ax2.legend(handles=legend_patches, loc='lower right',
               fontsize=9, framealpha=0.85)
    ax2.set_title(f"INSAT Rain / No-Rain Prediction\n{time_str}",
                  fontsize=12, fontweight='bold')
    # ← rain coverage annotation removed as requested

    # ASCII progress bar in suptitle
    progress = "█" * (frame_idx + 1) + "░" * (total_frames - frame_idx - 1)
    fig.suptitle(
        f"INSAT-3DR  |  Rain Detection          "
        f"[{frame_idx+1:02d}/{total_frames:02d}]  {progress}",
        fontsize=13, fontweight='bold', y=1.005
    )

    plt.tight_layout()
    frame_path = os.path.join(FRAMES_DIR, f"frame_{frame_idx:03d}.png")
    plt.savefig(frame_path, dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return frame_path

# ============================================================
# 5. LOAD MODEL
# ============================================================
model = load_model(MODEL_PATH, compile=False)
print("✅ Model loaded.")

# ============================================================
# 6. DISCOVER + SORT FILES
# ============================================================
pattern = os.path.join(INSAT_DIR, "3RIMG_*_L1C_SGP_V01R00.h5")
files   = sorted(glob.glob(pattern))

print(f"Found {len(files)} files:")
for f in files:
    print(" ", os.path.basename(f))

assert len(files) > 0, (
    f"No files found!\nDirectory : {INSAT_DIR}\nPattern   : {pattern}"
)

# Auto-detect date from first filename
first_ts   = parse_timestamp(files[0])
DATE_LABEL = first_ts.strftime("%d%b%Y").upper()   # e.g. "01AUG2025"

OUTPUT_GIF    = os.path.join(OUTPUT_DIR, f"rain_evolution_{DATE_LABEL}.gif")
OUTPUT_GIF_HQ = os.path.join(OUTPUT_DIR, f"rain_evolution_{DATE_LABEL}_hq.gif")

print(f"Date detected : {DATE_LABEL}")
print(f"Output folder : {OUTPUT_DIR}")
print(f"Output GIF    : {OUTPUT_GIF}")
print(f"Output GIF HQ : {OUTPUT_GIF_HQ}")

# ============================================================
# 7. MAIN LOOP — process all files, render frames
# ============================================================
frame_paths = []

for idx, fpath in enumerate(tqdm(files, desc="Processing INSAT files")):
    timestamp = parse_timestamp(fpath)
    if timestamp is None:
        print(f"  ⚠️  Skipping (can't parse timestamp): {fpath}")
        continue

    print(f"  [{idx+1:02d}/{len(files)}]  {timestamp.strftime('%H:%M UTC')}"
          f"  — {os.path.basename(fpath)}")

    rain_prob_map, tir1_bt, X, Y = process_file(fpath)
    lon, lat = get_lonlat(X, Y)

    fp = render_frame(rain_prob_map, tir1_bt, lon, lat,
                      timestamp, idx, len(files))
    frame_paths.append(fp)

print(f"\n✅ {len(frame_paths)} frames rendered → {FRAMES_DIR}")

# ============================================================
# 8. PREVIEW ALL FRAMES INLINE BEFORE SAVING
# ============================================================
print("Previewing frames...")

n      = len(frame_paths)
ncols  = 3
nrows  = (n + ncols - 1) // ncols   # auto row count for any number of files

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 6))
axes = axes.ravel()

for i, fp in enumerate(frame_paths):
    ts  = parse_timestamp(files[i])
    img = plt.imread(fp)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(
        f"Frame {i+1}  |  {ts.strftime('%H:%M UTC')}",
        fontsize=9, fontweight='bold'
    )

# hide unused axes if frames < grid slots
for j in range(n, len(axes)):
    axes[j].axis('off')

plt.suptitle(f"Frame Preview — {DATE_LABEL}  ({n} frames)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================
# 9. COMPILE → STANDARD GIF  (imageio)
# ============================================================
print("Compiling standard GIF...")

frames    = [imageio.imread(fp) for fp in frame_paths]
durations = [1.0 / FPS] * len(frames)
durations[0]  = 2.5
durations[-1] = 4.0

imageio.mimwrite(
    OUTPUT_GIF,
    frames,
    format='GIF',
    duration=durations,
    loop=0
)
print(f"✅ GIF saved  →  {OUTPUT_GIF}")
print(f"   Frames : {len(frames)}  |  FPS : {FPS}  |  "
      f"Duration : ~{len(frames)/FPS:.0f}s")

# ============================================================
# 10. COMPILE → HQ GIF  (Pillow — optimized, smaller file)
# ============================================================
print("Saving optimized HQ GIF with Pillow...")

pil_frames   = [Image.fromarray(imageio.imread(fp)) for fp in frame_paths]
durations_ms = [int(1000 / FPS)] * len(pil_frames)
durations_ms[0]  = 2500
durations_ms[-1] = 4000

pil_frames[0].save(
    OUTPUT_GIF_HQ,
    save_all=True,
    append_images=pil_frames[1:],
    duration=durations_ms,
    loop=0,
    optimize=True
)
print(f"✅ HQ GIF saved  →  {OUTPUT_GIF_HQ}")

# ============================================================
# 11. DISPLAY BOTH GIFS INLINE IN JUPYTER
# ============================================================
print("\n📽️  Standard GIF:")
display(IPImage(filename=OUTPUT_GIF))

print("📽️  HQ GIF:")
display(IPImage(filename=OUTPUT_GIF_HQ))

# INSAT Stratiform vs Convective

In [ ]:
"""
INSAT Two-Stage Pipeline GIF Generator  (RAM-efficient)

Stage 1 — Rain gate:      rain_no_rain.keras
  → marks every pixel as Rain or No-Rain

Stage 2 — Strat/Conv:     stratiform_vs_convective_layernorm.keras
  → runs ONLY on pixels flagged as Rain by Stage 1
  → No-Rain pixels are shown as white/clear on the final map

RAM strategy:
  - Patches built and predicted one small batch at a time
  - Gaussian-weighted accumulation (no bleeding across boundaries)
  - All intermediate arrays deleted + gc.collect() after each file
  - GIF compiled by streaming frames from disk (never held in memory)
"""

# ============================================================
# 0. IMPORTS
# ============================================================
import os, gc
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"
os.environ["TF_XLA_FLAGS"]          = "--tf_xla_enable_xla_devices=false"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import warnings
warnings.filterwarnings("ignore")

import re, glob
from datetime import datetime

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import tensorflow as tf
from tensorflow.keras.models import load_model
from pyproj import Proj, Transformer
from PIL import Image
from IPython.display import Image as IPImage, display
import imageio.v2 as imageio
from tqdm import tqdm

# ============================================================
# 1. GPU SETUP
# ============================================================
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU enabled: {[g.name for g in gpus]}")
else:
    print("No GPU - running on CPU")

# ============================================================
# 2. CONFIG
# ============================================================
RAIN_MODEL_PATH    = "rain_no_rain.keras"   # Stage 1 gate
SC_MODEL_PATH      = "/home/shruti/@best_layernorm_models/stratiform_vs_convective_layernorm.keras"

INSAT_DIR    = "/home/shruti/Downloads/insat/insat_aug_01"
OUTPUT_DIR   = "/home/shruti/Downloads/insat_gif_strat_conv"
FRAMES_DIR   = "/tmp/insat_strat_conv_frames/"
INDIA_EXTENT = [68, 97, 7, 37]
PATCH_SIZE   = 128
STRIDE       = 64
FPS          = 2
BATCH_SIZE   = 32

# Stage 1 threshold: pixel is "rain" if rain_prob > this
RAIN_THRESHOLD = 0.50   # standard 0.5 — model was trained for this

# Stage 2 threshold: among rainy pixels, label convective only if conv_prob > this
CONV_THRESHOLD = 0.55   # lower than before — no-rain mask already filters clear sky

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

# ============================================================
# 3. COLORMAPS
# ============================================================
# 4 categories: 0=No Data(geo invalid), 1=No Rain(white), 2=Stratiform(blue), 3=Convective(red)
FINAL_CMAP = ListedColormap(["#CCCCCC", "#FFFFFF", "#3A86FF", "#FF3A3A"])
bounds_fc  = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm_fc    = BoundaryNorm(bounds_fc, FINAL_CMAP.N)

legend_patches = [
    mpatches.Patch(color="#FF3A3A", label="Convective"),
    mpatches.Patch(color="#3A86FF", label="Stratiform"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
    mpatches.Patch(color="#CCCCCC", label="No Data"),
]

# ============================================================
# 4. GAUSSIAN KERNEL  (built once, shared by both models)
# ============================================================
def _make_gaussian_kernel(size, sigma_ratio=0.35):
    rng   = np.linspace(-1, 1, size, dtype=np.float32)   # exactly `size` points
    sigma = sigma_ratio
    g1d   = np.exp(-0.5 * (rng / sigma) ** 2)
    kernel = np.outer(g1d, g1d).astype(np.float32)
    kernel /= kernel.sum()
    return kernel   # shape: (size, size) guaranteed

_GAUSS_KERNEL = _make_gaussian_kernel(PATCH_SIZE)

# ============================================================
# 5. HELPERS
# ============================================================
def parse_timestamp(fpath):
    m = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{4})', os.path.basename(fpath))
    if m:
        return datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")
    return None


def _predict_map(model_obj, bt_norm, out_ch, H, W):
    """
    Shared Gaussian-weighted sliding-window predictor.
    model_obj : loaded keras model
    bt_norm   : (H, W, 4) float32 normalised BT
    out_ch    : which output channel index to accumulate (0 or 1)
    Returns   : (H, W) float32 probability map
    """
    prob_map  = np.zeros((H, W), dtype=np.float32)
    weight_map = np.zeros((H, W), dtype=np.float32)

    positions = [
        (i, j)
        for i in range(0, H - PATCH_SIZE + 1, STRIDE)
        for j in range(0, W - PATCH_SIZE + 1, STRIDE)
    ]

    for batch_start in range(0, len(positions), BATCH_SIZE):
        batch_pos = positions[batch_start : batch_start + BATCH_SIZE]
        patches = np.stack(
            [bt_norm[i:i+PATCH_SIZE, j:j+PATCH_SIZE] for i, j in batch_pos],
            axis=0
        )
        preds = model_obj.predict(patches, verbose=0)   # (B, 128, 128, N_classes)
        del patches

        for (i, j), pred in zip(batch_pos, preds):
            prob_map[i:i+PATCH_SIZE,   j:j+PATCH_SIZE] += pred[:, :, out_ch] * _GAUSS_KERNEL
            weight_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += _GAUSS_KERNEL
        del preds

    prob_map /= (weight_map + 1e-8)
    del weight_map
    return prob_map


def process_file(fpath):
    """
    Two-stage pipeline:
      Stage 1 — rain_model  → rain_prob_map  (channel 1 = rain)
      Stage 2 — sc_model    → strat/conv prob maps, but ONLY for rain pixels

    Returns:
      rain_mask     : (H, W) bool  — True where rain was detected
      strat_prob    : (H, W) float32  — 0 where no rain (meaningless, masked)
      conv_prob     : (H, W) float32  — 0 where no rain (meaningless, masked)
      tir1_bt       : (H, W) float32  — raw brightness temperature
      X, Y          : coordinate vectors
    """
    # ── 1. Read HDF5 ──────────────────────────────────────────
    with h5py.File(fpath, "r") as f:
        tir1 = f["IMG_TIR1"][0].astype(np.int32)
        tir2 = f["IMG_TIR2"][0].astype(np.int32)
        mir  = f["IMG_MIR"][0].astype(np.int32)
        wv   = f["IMG_WV"][0].astype(np.int32)
        ch0 = f["IMG_TIR1_TEMP"][:][tir1].astype(np.float32); del tir1
        ch1 = f["IMG_TIR2_TEMP"][:][tir2].astype(np.float32); del tir2
        ch2 = f["IMG_WV_TEMP"][:][wv].astype(np.float32);    del wv
        ch3 = f["IMG_MIR_TEMP"][:][mir].astype(np.float32);  del mir
        X   = f["X"][:]
        Y   = f["Y"][:]

    tir1_bt = ch0.copy()
    H, W    = ch0.shape

    # ── 2. Normalise in-place → (H, W, 4) float32 ────────────
    bt_norm = np.empty((H, W, 4), dtype=np.float32)
    for c_idx, ch in enumerate([ch0, ch1, ch2, ch3]):
        np.clip(ch, 180, 330, out=ch)
        ch -= 180
        ch /= 150.0
        bt_norm[:, :, c_idx] = ch
    del ch0, ch1, ch2, ch3
    gc.collect()

    # ── Stage 1: Rain / No-Rain gate ─────────────────────────
    print("    Stage 1: rain/no-rain ...", end=" ", flush=True)
    rain_prob = _predict_map(rain_model, bt_norm, out_ch=1, H=H, W=W)
    rain_mask = rain_prob > RAIN_THRESHOLD   # True = rainy pixel
    rain_pct  = rain_mask.mean() * 100
    print(f"done  ({rain_pct:.1f}% pixels flagged as rain)")
    del rain_prob
    gc.collect()

    # ── Stage 2: Strat/Conv — only on rainy pixels ────────────
    # We still run the sliding window over the full image for simplicity,
    # but the rain_mask is applied at classification time in render_frame.
    # This avoids the complexity of sub-image inference.
    print("    Stage 2: strat/conv ...", end=" ", flush=True)
    strat_prob = _predict_map(sc_model, bt_norm, out_ch=0, H=H, W=W)
    conv_prob  = _predict_map(sc_model, bt_norm, out_ch=1, H=H, W=W)
    print("done")

    del bt_norm
    gc.collect()

    return rain_mask, strat_prob, conv_prob, tir1_bt, X, Y


# lon/lat cache
_lon_lat = None

def get_lonlat(X, Y):
    global _lon_lat
    if _lon_lat is None:
        xx, yy      = np.meshgrid(X, Y)
        proj_geo    = Proj(proj="geos", h=35786023, lon_0=82.5, sweep="x")
        transformer = Transformer.from_proj(proj_geo, "epsg:4326", always_xy=True)
        lon, lat    = transformer.transform(xx, yy)
        _lon_lat    = (lon.astype(np.float32), lat.astype(np.float32))
        del xx, yy, lon, lat
        gc.collect()
        print("  (lon/lat grid computed and cached)")
    return _lon_lat


# ============================================================
# 6. SHARED MAP HELPER
# ============================================================
def _add_map_features(ax):
    ax.coastlines(resolution="10m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor="black")
    ax.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor="dimgray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.4,
                      color="gray", alpha=0.6, linestyle="--")
    gl.top_labels   = False
    gl.right_labels = False
    ax.set_extent(INDIA_EXTENT)


def _build_category_map(valid, rain_mask, conv_prob):
    """
    0 = geo invalid  (gray)
    1 = no rain      (white)
    2 = stratiform   (blue)
    3 = convective   (red)
    """
    cat = np.zeros(conv_prob.shape, dtype=np.float32)
    cat[valid & ~rain_mask]                              = 1   # no rain
    cat[valid & rain_mask & (conv_prob <= CONV_THRESHOLD)] = 2   # stratiform
    cat[valid & rain_mask & (conv_prob >  CONV_THRESHOLD)] = 3   # convective
    return cat


# ── GIF A colormaps ──────────────────────────────────────────
RAIN_CMAP = ListedColormap(["#FFFFFF", "#2196F3"])
RAIN_LEGEND = [
    mpatches.Patch(color="#2196F3", label="Rain"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
]


# ============================================================
# 6a. RENDER FRAME — GIF A  (TIR1 | Rain/No-Rain | Strat/Conv)
# ============================================================
def render_frame_A(rain_mask, conv_prob, tir1_bt, lon, lat,
                   timestamp, frame_idx, total_frames):

    valid     = np.isfinite(lat) & np.isfinite(lon)
    lon_clean = np.where(valid, lon, 0.0)
    lat_clean = np.where(valid, lat, 0.0)
    time_str  = timestamp.strftime("%d %b %Y  %H:%M UTC")

    cat = _build_category_map(valid, rain_mask, conv_prob)

    tir1_plot = np.ma.array(tir1_bt,                      mask=~valid)
    rain_plot = np.ma.array(rain_mask.astype(np.float32), mask=~valid)
    cat_plot  = np.ma.array(cat,                           mask=~valid)
    del cat

    fig, axes = plt.subplots(1, 3, figsize=(21, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    # Panel 1: TIR1 BT
    im1 = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                              cmap="RdYlBu_r", vmin=180, vmax=320,
                              transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[0])
    plt.colorbar(im1, ax=axes[0], orientation="vertical",
                 pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    # Panel 2: Rain / No-Rain
    axes[1].pcolormesh(lon_clean, lat_clean, rain_plot,
                       cmap=RAIN_CMAP, vmin=0, vmax=1,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[1])
    axes[1].legend(handles=RAIN_LEGEND, loc="lower right",
                   fontsize=8, framealpha=0.85)
    axes[1].set_title(f"Rain / No-Rain\n{time_str}", fontsize=10, fontweight="bold")

    # Panel 3: Strat / Conv / No-Rain
    axes[2].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=FINAL_CMAP, norm=norm_fc,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[2])
    axes[2].legend(handles=legend_patches, loc="lower right",
                   fontsize=8, framealpha=0.85)
    axes[2].set_title(f"Stratiform / Convective\n{time_str}", fontsize=10, fontweight="bold")

    done = "█" * (frame_idx + 1) + "░" * (total_frames - frame_idx - 1)
    fig.suptitle(f"INSAT-3DR  |  GIF-A: TIR1 | Rain Gate | Classification"
                 f"     [{frame_idx+1:02d}/{total_frames:02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)

    plt.tight_layout()
    path = os.path.join(FRAMES_DIR, f"A_{frame_idx:03d}.png")
    plt.savefig(path, dpi=120, facecolor="white")
    plt.close(fig)
    del tir1_plot, rain_plot, cat_plot, lon_clean, lat_clean, valid, fig, axes
    gc.collect()
    return path


# ============================================================
# 6b. RENDER FRAME — GIF B  (TIR1 | Strat/Conv/No-Rain)
# ============================================================
def render_frame_B(rain_mask, conv_prob, tir1_bt, lon, lat,
                   timestamp, frame_idx, total_frames):

    valid     = np.isfinite(lat) & np.isfinite(lon)
    lon_clean = np.where(valid, lon, 0.0)
    lat_clean = np.where(valid, lat, 0.0)
    time_str  = timestamp.strftime("%d %b %Y  %H:%M UTC")

    cat = _build_category_map(valid, rain_mask, conv_prob)

    tir1_plot = np.ma.array(tir1_bt, mask=~valid)
    cat_plot  = np.ma.array(cat,     mask=~valid)
    del cat

    fig, axes = plt.subplots(1, 2, figsize=(14, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    # Panel 1: TIR1 BT
    im1 = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                              cmap="RdYlBu_r", vmin=180, vmax=320,
                              transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[0])
    plt.colorbar(im1, ax=axes[0], orientation="vertical",
                 pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    # Panel 2: Strat / Conv / No-Rain
    axes[1].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=FINAL_CMAP, norm=norm_fc,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[1])
    axes[1].legend(handles=legend_patches, loc="lower right",
                   fontsize=8, framealpha=0.85)
    axes[1].set_title(f"No Rain / Stratiform / Convective\n{time_str}", fontsize=10, fontweight="bold")

    done = "█" * (frame_idx + 1) + "░" * (total_frames - frame_idx - 1)
    fig.suptitle(f"INSAT-3DR  |  GIF-B: TIR1 | Classification"
                 f"     [{frame_idx+1:02d}/{total_frames:02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)

    plt.tight_layout()
    path = os.path.join(FRAMES_DIR, f"B_{frame_idx:03d}.png")
    plt.savefig(path, dpi=120, facecolor="white")
    plt.close(fig)
    del tir1_plot, cat_plot, lon_clean, lat_clean, valid, fig, axes
    gc.collect()
    return path


# ============================================================
# 7. LOAD BOTH MODELS
# ============================================================
print("Loading rain/no-rain model (Stage 1)...")
rain_model = load_model(RAIN_MODEL_PATH, compile=False)
print("Loading strat/conv model (Stage 2)...")
sc_model   = load_model(SC_MODEL_PATH, compile=False)
print("Both models loaded.\n")

# ============================================================
# 8. DISCOVER + SORT FILES
# ============================================================
pattern = os.path.join(INSAT_DIR, "3RIMG_*_L1C_SGP_V01R00.h5")
files   = sorted(glob.glob(pattern))

print(f"Found {len(files)} files:")
for f in files:
    print(" ", os.path.basename(f))

assert len(files) > 0, f"No files found!\nDirectory: {INSAT_DIR}\nPattern: {pattern}"

first_ts   = parse_timestamp(files[0])
DATE_LABEL = first_ts.strftime("%d%b%Y").upper()
OUTPUT_GIF    = os.path.join(OUTPUT_DIR, f"two_stage_{DATE_LABEL}.gif")
OUTPUT_GIF_HQ = os.path.join(OUTPUT_DIR, f"two_stage_{DATE_LABEL}_hq.gif")

print(f"\nDate       : {DATE_LABEL}")
print(f"Output dir : {OUTPUT_DIR}\n")

# ============================================================
# 9. MAIN LOOP  — renders both GIF-A and GIF-B frames per file
# ============================================================
frames_A = []
frames_B = []

for idx, fpath in enumerate(tqdm(files, desc="Processing files")):
    timestamp = parse_timestamp(fpath)
    if timestamp is None:
        print(f"  Skipping (bad timestamp): {fpath}")
        continue

    print(f"  [{idx+1:02d}/{len(files)}]  {timestamp.strftime('%H:%M UTC')}  — {os.path.basename(fpath)}")

    rain_mask, strat_prob, conv_prob, tir1_bt, X, Y = process_file(fpath)
    lon, lat = get_lonlat(X, Y)

    frames_A.append(render_frame_A(rain_mask, conv_prob, tir1_bt, lon, lat,
                                   timestamp, idx, len(files)))
    frames_B.append(render_frame_B(rain_mask, conv_prob, tir1_bt, lon, lat,
                                   timestamp, idx, len(files)))

    del rain_mask, strat_prob, conv_prob, tir1_bt
    gc.collect()

# ============================================================
# 10. COMPILE GIFs  (shared helper — streams frames one at a time)
# ============================================================
def _compile_gif(frame_list, out_hq, out_std, label):
    n = len(frame_list)
    if n == 0:
        print(f"No frames for {label}, skipping.")
        return

    frame_ms = [int(1000 / FPS)] * n
    frame_ms[0]  = 2500
    frame_ms[-1] = 4000

    def _pil(path, sz=None):
        img = Image.open(path).convert("RGBA")
        if sz and img.size != sz:
            img = img.resize(sz, Image.LANCZOS)
        return img

    print(f"Saving {label} HQ GIF...")
    first = _pil(frame_list[0])
    sz    = first.size
    first.save(out_hq, save_all=True,
               append_images=(_pil(fp, sz) for fp in frame_list[1:]),
               duration=frame_ms, loop=0, optimize=True)
    del first
    gc.collect()
    print(f"  -> {out_hq}")

    print(f"Saving {label} standard GIF...")
    tw, th = sz
    with imageio.get_writer(out_std, mode="I", loop=0) as writer:
        for fp in frame_list:
            frame = imageio.imread(fp)
            if frame.shape[:2] != (th, tw):
                frame = np.array(Image.fromarray(frame).resize((tw, th), Image.LANCZOS))
            writer.append_data(frame)
            del frame
    gc.collect()
    print(f"  -> {out_std}")


GIF_A_HQ  = os.path.join(OUTPUT_DIR, f"A_tir1_rain_sc_{DATE_LABEL}_hq.gif")
GIF_A_STD = os.path.join(OUTPUT_DIR, f"A_tir1_rain_sc_{DATE_LABEL}.gif")
GIF_B_HQ  = os.path.join(OUTPUT_DIR, f"B_tir1_sc_{DATE_LABEL}_hq.gif")
GIF_B_STD = os.path.join(OUTPUT_DIR, f"B_tir1_sc_{DATE_LABEL}.gif")

_compile_gif(frames_A, GIF_A_HQ, GIF_A_STD, "GIF-A (TIR1 | Rain | Strat/Conv)")
_compile_gif(frames_B, GIF_B_HQ, GIF_B_STD, "GIF-B (TIR1 | Strat/Conv)")

# ============================================================
# 11. DISPLAY IN JUPYTER
# ============================================================
print("\n=== GIF-A: TIR1 | Rain/No-Rain | Strat/Conv ===")
display(IPImage(filename=GIF_A_STD))
print("\n=== GIF-B: TIR1 | Strat/Conv/No-Rain ===")
display(IPImage(filename=GIF_B_STD))

# fourclasses_layernorm.keras

In [ ]:
"""
INSAT Three-Stage Pipeline GIF Generator  (RAM-efficient)

Stage 1 — Rain gate:    rain_no_rain.keras        → Rain / No-Rain
Stage 2 — Strat/Conv:   strat_conv 2-class model  → Stratiform vs Convective
Stage 3 — Four-class:   fourclasses_layernorm.keras
            0 = Stratiform
            1 = Deep Convective
            2 = Shallow Convective (isolated)
            3 = Shallow Convective (non-isolated)

GIF-A (3 panels): TIR1 | Rain/No-Rain | Rain sub-classes (strat + 3 conv sub-types + no-rain)
GIF-B (2 panels): TIR1 | Four-class + No-Rain
"""

# ============================================================
# 0. IMPORTS
# ============================================================
import os, gc
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"
os.environ["TF_XLA_FLAGS"]          = "--tf_xla_enable_xla_devices=false"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import warnings
warnings.filterwarnings("ignore")

import re, glob
from datetime import datetime

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import tensorflow as tf
from tensorflow.keras.models import load_model
from pyproj import Proj, Transformer
from PIL import Image
from IPython.display import Image as IPImage, display
import imageio.v2 as imageio
from tqdm import tqdm

# # ============================================================
# # 1. GPU SETUP
# # ============================================================
# gpus = tf.config.list_physical_devices("GPU")
# if gpus:
#     for gpu in gpus:
#         tf.config.experimental.set_memory_growth(gpu, True)
#     print(f"GPU enabled: {[g.name for g in gpus]}")
# else:
#     print("No GPU - running on CPU")

# ============================================================
# 2. CONFIG
# ============================================================
RAIN_MODEL_PATH    = "/home/shruti/@best_layernorm_models/rain_no_rain.keras"   # Stage 1 gate
SC_MODEL_PATH      = "/home/shruti/@best_layernorm_models/stratiform_vs_convective_layernorm.keras"  # unused now but kept
FC_MODEL_PATH      = "/home/shruti/@best_layernorm_models/fourclasses_layernorm.keras"   # Stage 3: 4-class

INSAT_DIR    = "/home/shruti/Downloads/insat/insat_aug_01"
OUTPUT_DIR   = "/home/shruti/Downloads/insat_gif_strat_conv"
FRAMES_DIR   = "/tmp/insat_strat_conv_frames/"
INDIA_EXTENT = [68, 97, 7, 37]
PATCH_SIZE   = 128
STRIDE       = 64
FPS          = 2
BATCH_SIZE   = 8   # 4-class model outputs (B,128,128,4) — keep small to avoid OOM
                    # lower to 8 if kernel still dies; raise to 32 if GPU has spare VRAM
STRIDE       = 96   # larger stride = fewer patches = less RAM (was 64); tune 64-128

# Stage 1 threshold: pixel is "rain" if rain_prob > this
RAIN_THRESHOLD = 0.50   # standard 0.5 — model was trained for this

# Stage 2 threshold: among rainy pixels, label convective only if conv_prob > this
CONV_THRESHOLD = 0.55   # lower than before — no-rain mask already filters clear sky

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

# ============================================================
# 3. COLORMAPS
# ============================================================
# ── GIF-A: Rain sub-classes panel ───────────────────────────
# 0=No Data(gray) 1=No Rain(white) 2=Stratiform(blue)
# 3=Deep Conv(red) 4=Shallow Isolated(orange) 5=Shallow Non-iso(yellow)
SUBA_CMAP   = ListedColormap(["#CCCCCC", "#FFFFFF", "#3A86FF",
                               "#D32F2F", "#FF7043", "#FFC107"])
bounds_suba = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm_suba   = BoundaryNorm(bounds_suba, SUBA_CMAP.N)

legend_suba = [
    mpatches.Patch(color="#3A86FF", label="Stratiform"),
    mpatches.Patch(color="#D32F2F", label="Deep Convective"),
    mpatches.Patch(color="#FF7043", label="Shallow Conv (isolated)"),
    mpatches.Patch(color="#FFC107", label="Shallow Conv (non-iso)"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
    mpatches.Patch(color="#CCCCCC", label="No Data"),
]

# ── GIF-B: compact 4-class + no-rain panel ──────────────────
# same as SUBA but kept as alias for clarity
FINAL_CMAP  = SUBA_CMAP
norm_fc     = norm_suba
legend_patches = legend_suba

# ── Rain gate colormap (shared) ─────────────────────────────
RAIN_CMAP = ListedColormap(["#FFFFFF", "#2196F3"])
RAIN_LEGEND = [
    mpatches.Patch(color="#2196F3", label="Rain"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
]

# ============================================================
# 4. GAUSSIAN KERNEL  (built once, shared by both models)
# ============================================================
def _make_gaussian_kernel(size, sigma_ratio=0.35):
    rng   = np.linspace(-1, 1, size, dtype=np.float32)   # exactly `size` points
    sigma = sigma_ratio
    g1d   = np.exp(-0.5 * (rng / sigma) ** 2)
    kernel = np.outer(g1d, g1d).astype(np.float32)
    kernel /= kernel.sum()
    return kernel   # shape: (size, size) guaranteed

_GAUSS_KERNEL = _make_gaussian_kernel(PATCH_SIZE)

# ============================================================
# 5. HELPERS
# ============================================================
def parse_timestamp(fpath):
    m = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{4})', os.path.basename(fpath))
    if m:
        return datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")
    return None


def _predict_map(model_obj, bt_norm, out_ch, H, W):
    """
    Shared Gaussian-weighted sliding-window predictor.
    model_obj : loaded keras model
    bt_norm   : (H, W, 4) float32 normalised BT
    out_ch    : which output channel index to accumulate (0 or 1)
    Returns   : (H, W) float32 probability map
    """
    prob_map  = np.zeros((H, W), dtype=np.float32)
    weight_map = np.zeros((H, W), dtype=np.float32)

    positions = [
        (i, j)
        for i in range(0, H - PATCH_SIZE + 1, STRIDE)
        for j in range(0, W - PATCH_SIZE + 1, STRIDE)
    ]

    for batch_start in range(0, len(positions), BATCH_SIZE):
        batch_pos = positions[batch_start : batch_start + BATCH_SIZE]
        patches = np.stack(
            [bt_norm[i:i+PATCH_SIZE, j:j+PATCH_SIZE] for i, j in batch_pos],
            axis=0
        )
        preds = model_obj.predict(patches, verbose=0)   # (B, 128, 128, N_classes)
        del patches

        for (i, j), pred in zip(batch_pos, preds):
            prob_map[i:i+PATCH_SIZE,   j:j+PATCH_SIZE] += pred[:, :, out_ch] * _GAUSS_KERNEL
            weight_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += _GAUSS_KERNEL
        del preds
        gc.collect()   # free GPU output buffer after every batch

    prob_map /= (weight_map + 1e-8)
    del weight_map
    return prob_map


def process_file(fpath):
    """
    Two-stage pipeline:
      Stage 1 — rain_model  → rain_prob_map  (channel 1 = rain)
      Stage 2 — sc_model    → strat/conv prob maps, but ONLY for rain pixels

    Returns:
      rain_mask  : (H, W) bool     — True where rain was detected
      fc_probs   : (H, W, 4) float32 — softmax probs for 4 classes (rainy pixels only meaningful)
      tir1_bt    : (H, W) float32  — raw brightness temperature
      X, Y       : coordinate vectors
    """
    # ── 1. Read HDF5 ──────────────────────────────────────────
    with h5py.File(fpath, "r") as f:
        tir1 = f["IMG_TIR1"][0].astype(np.int32)
        tir2 = f["IMG_TIR2"][0].astype(np.int32)
        mir  = f["IMG_MIR"][0].astype(np.int32)
        wv   = f["IMG_WV"][0].astype(np.int32)
        ch0 = f["IMG_TIR1_TEMP"][:][tir1].astype(np.float32); del tir1
        ch1 = f["IMG_TIR2_TEMP"][:][tir2].astype(np.float32); del tir2
        ch2 = f["IMG_WV_TEMP"][:][wv].astype(np.float32);    del wv
        ch3 = f["IMG_MIR_TEMP"][:][mir].astype(np.float32);  del mir
        X   = f["X"][:]
        Y   = f["Y"][:]

    tir1_bt = ch0.copy()
    H, W    = ch0.shape

    # ── 2. Normalise in-place → (H, W, 4) float32 ────────────
    bt_norm = np.empty((H, W, 4), dtype=np.float32)
    for c_idx, ch in enumerate([ch0, ch1, ch2, ch3]):
        np.clip(ch, 180, 330, out=ch)
        ch -= 180
        ch /= 150.0
        bt_norm[:, :, c_idx] = ch
    del ch0, ch1, ch2, ch3
    gc.collect()

    # ── Stage 1: Rain / No-Rain gate ─────────────────────────
    print("    Stage 1: rain/no-rain ...", end=" ", flush=True)
    rain_prob = _predict_map(rain_model, bt_norm, out_ch=1, H=H, W=W)
    rain_mask = rain_prob > RAIN_THRESHOLD   # True = rainy pixel
    rain_pct  = rain_mask.mean() * 100
    print(f"done  ({rain_pct:.1f}% pixels flagged as rain)")
    del rain_prob
    gc.collect()

    # ── Stage 2: Four-class model ────────────────────────────────────────
    # RAM strategy:
    #   - BATCH_SIZE kept small (defined in CONFIG)
    #   - preds cast to float16 immediately after inference → half the memory
    #   - accumulation arrays are float32 (precision where it matters)
    #   - fc_probs (H,W,4) float16 returned — ~4x smaller than float32
    #   - weight_map deleted as soon as normalisation is done
    #   - bt_norm deleted before return
    print("    Stage 2: four-class ...", end=" ", flush=True)

    H2, W2 = bt_norm.shape[:2]
    fc_probs   = np.zeros((H2, W2, 4), dtype=np.float32)
    weight_map = np.zeros((H2, W2),    dtype=np.float32)

    positions = [
        (i, j)
        for i in range(0, H2 - PATCH_SIZE + 1, STRIDE)
        for j in range(0, W2 - PATCH_SIZE + 1, STRIDE)
    ]

    for batch_start in range(0, len(positions), BATCH_SIZE):
        batch_pos = positions[batch_start : batch_start + BATCH_SIZE]

        # build only this batch — never the full patch array
        patches = np.stack(
            [bt_norm[i:i+PATCH_SIZE, j:j+PATCH_SIZE] for i, j in batch_pos],
            axis=0
        )   # (B, 128, 128, 4) float32

        # cast predictions to float16 immediately → half the memory
        preds = fc_model.predict(patches, verbose=0).astype(np.float16)
        del patches

        for (i, j), pred in zip(batch_pos, preds):
            # cast back to float32 only for the accumulation slice
            fc_probs[i:i+PATCH_SIZE, j:j+PATCH_SIZE, :] += (
                pred.astype(np.float32) * _GAUSS_KERNEL[:, :, None]
            )
            weight_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += _GAUSS_KERNEL
        del preds
        gc.collect()   # free after every batch

    # normalise then convert to float16 to halve return size
    fc_probs /= (weight_map[:, :, None] + 1e-8)
    fc_probs  = fc_probs.astype(np.float16)
    del weight_map
    print("done")

    del bt_norm
    gc.collect()

    return rain_mask, fc_probs, tir1_bt, X, Y


# lon/lat cache
_lon_lat = None

def get_lonlat(X, Y):
    global _lon_lat
    if _lon_lat is None:
        xx, yy      = np.meshgrid(X, Y)
        proj_geo    = Proj(proj="geos", h=35786023, lon_0=82.5, sweep="x")
        transformer = Transformer.from_proj(proj_geo, "epsg:4326", always_xy=True)
        lon, lat    = transformer.transform(xx, yy)
        _lon_lat    = (lon.astype(np.float32), lat.astype(np.float32))
        del xx, yy, lon, lat
        gc.collect()
        print("  (lon/lat grid computed and cached)")
    return _lon_lat


# ============================================================
# 6. SHARED MAP HELPERS
# ============================================================
def _add_map_features(ax):
    ax.coastlines(resolution="10m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor="black")
    ax.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor="dimgray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.4,
                      color="gray", alpha=0.6, linestyle="--")
    gl.top_labels   = False
    gl.right_labels = False
    ax.set_extent(INDIA_EXTENT)


def _build_fourclass_map(valid, rain_mask, fc_probs):
    """
    Build 6-value category map from four-class model output.

    Four-class model channels:
      0 = Stratiform
      1 = Deep Convective
      2 = Shallow Convective (isolated)
      3 = Shallow Convective (non-isolated)

    Output map values:
      0 = geo invalid          (gray   #CCCCCC)
      1 = No Rain              (white  #FFFFFF)
      2 = Stratiform           (blue   #3A86FF)
      3 = Deep Convective      (red    #D32F2F)
      4 = Shallow Isolated     (orange #FF7043)
      5 = Shallow Non-iso      (yellow #FFC107)
    """
    # argmax over 4 channels → class index 0-3
    pred_class = np.argmax(fc_probs, axis=-1).astype(np.float32)  # (H, W)

    cat = np.zeros(pred_class.shape, dtype=np.float32)   # 0 = geo invalid
    cat[valid & ~rain_mask] = 1                           # no rain
    # rainy pixels: shift model class 0-3 → map values 2-5
    rainy = valid & rain_mask
    cat[rainy] = pred_class[rainy] + 2
    return cat


# ============================================================
# 6a. RENDER FRAME — GIF A
#     3 panels: TIR1 | Rain/No-Rain | Rain sub-classes
# ============================================================
def render_frame_A(rain_mask, fc_probs, tir1_bt, lon, lat,
                   timestamp, frame_idx, total_frames):

    valid     = np.isfinite(lat) & np.isfinite(lon)
    lon_clean = np.where(valid, lon, 0.0)
    lat_clean = np.where(valid, lat, 0.0)
    time_str  = timestamp.strftime("%d %b %Y  %H:%M UTC")

    cat = _build_fourclass_map(valid, rain_mask, fc_probs)

    tir1_plot = np.ma.array(tir1_bt,                      mask=~valid)
    rain_plot = np.ma.array(rain_mask.astype(np.float32), mask=~valid)
    cat_plot  = np.ma.array(cat,                           mask=~valid)
    del cat

    fig, axes = plt.subplots(1, 3, figsize=(21, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    im1 = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                              cmap="RdYlBu_r", vmin=180, vmax=320,
                              transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[0])
    plt.colorbar(im1, ax=axes[0], orientation="vertical",
                 pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    axes[1].pcolormesh(lon_clean, lat_clean, rain_plot,
                       cmap=RAIN_CMAP, vmin=0, vmax=1,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[1])
    axes[1].legend(handles=RAIN_LEGEND, loc="lower right", fontsize=8, framealpha=0.85)
    axes[1].set_title(f"Stage 1: Rain / No-Rain\n{time_str}", fontsize=10, fontweight="bold")

    axes[2].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=SUBA_CMAP, norm=norm_suba,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[2])
    axes[2].legend(handles=legend_suba, loc="lower right", fontsize=7, framealpha=0.85)
    axes[2].set_title(f"Rain Sub-classes\n{time_str}", fontsize=10, fontweight="bold")

    done = "\u2588" * (frame_idx + 1) + "\u2591" * (total_frames - frame_idx - 1)
    fig.suptitle(f"INSAT-3DR  |  GIF-A: TIR1 | Rain Gate | Sub-classes"
                 f"     [{frame_idx+1:02d}/{total_frames:02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)

    plt.tight_layout()
    path = os.path.join(FRAMES_DIR, f"A_{frame_idx:03d}.png")
    plt.savefig(path, dpi=120, facecolor="white")
    plt.close(fig)
    del tir1_plot, rain_plot, cat_plot, lon_clean, lat_clean, valid, fig, axes
    gc.collect()
    return path


# ============================================================
# 6b. RENDER FRAME — GIF B
#     2 panels: TIR1 | Four-class + No-Rain
# ============================================================
def render_frame_B(rain_mask, fc_probs, tir1_bt, lon, lat,
                   timestamp, frame_idx, total_frames):

    valid     = np.isfinite(lat) & np.isfinite(lon)
    lon_clean = np.where(valid, lon, 0.0)
    lat_clean = np.where(valid, lat, 0.0)
    time_str  = timestamp.strftime("%d %b %Y  %H:%M UTC")

    cat = _build_fourclass_map(valid, rain_mask, fc_probs)

    tir1_plot = np.ma.array(tir1_bt, mask=~valid)
    cat_plot  = np.ma.array(cat,     mask=~valid)
    del cat

    fig, axes = plt.subplots(1, 2, figsize=(14, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    im1 = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                              cmap="RdYlBu_r", vmin=180, vmax=320,
                              transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[0])
    plt.colorbar(im1, ax=axes[0], orientation="vertical",
                 pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    axes[1].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=SUBA_CMAP, norm=norm_suba,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map_features(axes[1])
    axes[1].legend(handles=legend_suba, loc="lower right", fontsize=7, framealpha=0.85)
    axes[1].set_title(f"Four-class + No Rain\n{time_str}", fontsize=10, fontweight="bold")

    done = "\u2588" * (frame_idx + 1) + "\u2591" * (total_frames - frame_idx - 1)
    fig.suptitle(f"INSAT-3DR  |  GIF-B: TIR1 | Four-class Classification"
                 f"     [{frame_idx+1:02d}/{total_frames:02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)

    plt.tight_layout()
    path = os.path.join(FRAMES_DIR, f"B_{frame_idx:03d}.png")
    plt.savefig(path, dpi=120, facecolor="white")
    plt.close(fig)
    del tir1_plot, cat_plot, lon_clean, lat_clean, valid, fig, axes
    gc.collect()
    return path
# ============================================================
# 7. LOAD MODELS
# ============================================================
print("Loading rain/no-rain model (Stage 1)...")
rain_model = load_model(RAIN_MODEL_PATH, compile=False)
print("Loading four-class model (Stage 2)...")
fc_model   = load_model(FC_MODEL_PATH,   compile=False)
print("Both models loaded.\n")

# ============================================================
# 8. DISCOVER + SORT FILES
# ============================================================
pattern = os.path.join(INSAT_DIR, "3RIMG_*_L1C_SGP_V01R00.h5")
files   = sorted(glob.glob(pattern))

print(f"Found {len(files)} files:")
for f in files:
    print(" ", os.path.basename(f))

assert len(files) > 0, f"No files found!\nDirectory: {INSAT_DIR}\nPattern: {pattern}"

first_ts   = parse_timestamp(files[0])
DATE_LABEL = first_ts.strftime("%d%b%Y").upper()
OUTPUT_GIF    = os.path.join(OUTPUT_DIR, f"two_stage_{DATE_LABEL}.gif")
OUTPUT_GIF_HQ = os.path.join(OUTPUT_DIR, f"two_stage_{DATE_LABEL}_hq.gif")

print(f"\nDate       : {DATE_LABEL}")
print(f"Output dir : {OUTPUT_DIR}\n")

# ============================================================
# 9. MAIN LOOP  — renders both GIF-A and GIF-B frames per file
# ============================================================
frames_A = []
frames_B = []

for idx, fpath in enumerate(tqdm(files, desc="Processing files")):
    timestamp = parse_timestamp(fpath)
    if timestamp is None:
        print(f"  Skipping (bad timestamp): {fpath}")
        continue

    print(f"  [{idx+1:02d}/{len(files)}]  {timestamp.strftime('%H:%M UTC')}  — {os.path.basename(fpath)}")

    rain_mask, fc_probs, tir1_bt, X, Y = process_file(fpath)
    lon, lat = get_lonlat(X, Y)

    frames_A.append(render_frame_A(rain_mask, fc_probs, tir1_bt, lon, lat,
                                   timestamp, idx, len(files)))
    frames_B.append(render_frame_B(rain_mask, fc_probs, tir1_bt, lon, lat,
                                   timestamp, idx, len(files)))

    del rain_mask, fc_probs, tir1_bt
    gc.collect()

# ============================================================
# 10. COMPILE GIFs  (shared helper — streams frames one at a time)
# ============================================================
def _compile_gif(frame_list, out_hq, out_std, label):
    n = len(frame_list)
    if n == 0:
        print(f"No frames for {label}, skipping.")
        return

    frame_ms = [int(1000 / FPS)] * n
    frame_ms[0]  = 2500
    frame_ms[-1] = 4000

    def _pil(path, sz=None):
        img = Image.open(path).convert("RGBA")
        if sz and img.size != sz:
            img = img.resize(sz, Image.LANCZOS)
        return img

    print(f"Saving {label} HQ GIF...")
    first = _pil(frame_list[0])
    sz    = first.size
    first.save(out_hq, save_all=True,
               append_images=(_pil(fp, sz) for fp in frame_list[1:]),
               duration=frame_ms, loop=0, optimize=True)
    del first
    gc.collect()
    print(f"  -> {out_hq}")

    print(f"Saving {label} standard GIF...")
    tw, th = sz
    with imageio.get_writer(out_std, mode="I", loop=0) as writer:
        for fp in frame_list:
            frame = imageio.imread(fp)
            if frame.shape[:2] != (th, tw):
                frame = np.array(Image.fromarray(frame).resize((tw, th), Image.LANCZOS))
            writer.append_data(frame)
            del frame
    gc.collect()
    print(f"  -> {out_std}")


GIF_A_HQ  = os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}_hq.gif")
GIF_A_STD = os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}.gif")
GIF_B_HQ  = os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}_hq.gif")
GIF_B_STD = os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}.gif")

_compile_gif(frames_A, GIF_A_HQ, GIF_A_STD, "GIF-A (TIR1 | Rain | Strat/Conv)")
_compile_gif(frames_B, GIF_B_HQ, GIF_B_STD, "GIF-B (TIR1 | Strat/Conv)")

# ============================================================
# 11. DISPLAY IN JUPYTER
# ============================================================
print("\n=== GIF-A: TIR1 | Rain/No-Rain | Strat/Conv ===")
display(IPImage(filename=GIF_A_STD))
print("\n=== GIF-B: TIR1 | Strat/Conv/No-Rain ===")
display(IPImage(filename=GIF_B_STD))

In [ ]:
"""
INSAT Two-Stage GIF Generator — Minimal RAM Edition

Key strategy to avoid kernel death:
  1. Load rain model → run all 47 files → save rain masks to disk → DELETE model
  2. Load four-class model → run all 47 files → save fc_probs to disk → DELETE model
  3. Load frames one at a time from disk → render → save PNG → delete arrays
  4. Compile GIFs by streaming PNGs from disk — never hold all frames in RAM

Only ONE model lives in memory at a time.
"""

# ============================================================
# 0. IMPORTS + FORCE CPU BEFORE TF LOADS
# ============================================================
import os, gc
os.environ["CUDA_VISIBLE_DEVICES"]   = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"]   = "3"
os.environ["TF_XLA_FLAGS"]           = "--tf_xla_enable_xla_devices=false"
os.environ["TF_ENABLE_ONEDNN_OPTS"]  = "0"
os.environ["OMP_NUM_THREADS"]        = "4"
os.environ["TF_NUM_INTRAOP_THREADS"] = "4"
os.environ["TF_NUM_INTEROP_THREADS"] = "2"

import warnings
warnings.filterwarnings("ignore")

import re, glob
from datetime import datetime

import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — no display buffer in RAM
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from pyproj import Proj, Transformer
from PIL import Image
from IPython.display import Image as IPImage, display
import imageio.v2 as imageio
from tqdm import tqdm

# ============================================================
# 1. CONFIG
# ============================================================
RAIN_MODEL_PATH = "/home/shruti/Prep_project/notebooks/models/4_inputs_BT_only/rain_norain_BT_New.keras"
FC_MODEL_PATH   = "/home/shruti/Prep_project/notebooks/models/4_inputs_BT_only/fourclasses_layernorm_BT_only.keras"

INSAT_DIR    = "/home/shruti/Documents/insat_data"
OUTPUT_DIR   = "/home/shruti/Downloads/insat_gif_strat_conv2"
FRAMES_DIR   = "/tmp/insat_frames/"
CACHE_DIR    = "/tmp/insat_cache/"   # rain masks + fc_probs saved here between stages

INDIA_EXTENT = [68, 97, 7, 37]
PATCH_SIZE   = 128
STRIDE       = 96   # no overlap → fewest patches, least RAM
                     # set to 96 for slight smoothing if results look blocky
FPS          = 2
BATCH_SIZE   = 4     # very small — 4 × (128×128×4) float32 ≈ 4 MB per batch

RAIN_THRESHOLD = 0.50

for d in [OUTPUT_DIR, FRAMES_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# 2. COLORMAPS  (no models needed — safe to define now)
# ============================================================
SUBA_CMAP   = ListedColormap(["#CCCCCC", "#FFFFFF", "#3A86FF",
                               "#D32F2F", "#FF7043", "#FFC107"])
bounds_suba = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm_suba   = BoundaryNorm(bounds_suba, SUBA_CMAP.N)

legend_suba = [
    mpatches.Patch(color="#3A86FF", label="Stratiform"),
    mpatches.Patch(color="#D32F2F", label="Deep Convective"),
    mpatches.Patch(color="#FF7043", label="Shallow Conv (isolated)"),
    mpatches.Patch(color="#FFC107", label="Shallow Conv (non-iso)"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
    mpatches.Patch(color="#CCCCCC", label="No Data"),
]
RAIN_CMAP   = ListedColormap(["#FFFFFF", "#2196F3"])
RAIN_LEGEND = [
    mpatches.Patch(color="#2196F3", label="Rain"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
]

# ============================================================
# 3. HELPERS
# ============================================================
def parse_timestamp(fpath):
    m = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{4})', os.path.basename(fpath))
    if m:
        return datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")
    return None


def _make_gaussian_kernel(size):
    rng    = np.linspace(-1, 1, size, dtype=np.float32)
    g1d    = np.exp(-0.5 * (rng / 0.35) ** 2)
    kernel = np.outer(g1d, g1d).astype(np.float32)
    kernel /= kernel.sum()
    return kernel

_KERN = _make_gaussian_kernel(PATCH_SIZE)


def read_hdf5(fpath):
    """Read one HDF5 file, return bt_norm (H,W,4) float32, tir1_bt, X, Y."""
    with h5py.File(fpath, "r") as f:
        tir1 = f["IMG_TIR1"][0].astype(np.int32)
        tir2 = f["IMG_TIR2"][0].astype(np.int32)
        mir  = f["IMG_MIR"][0].astype(np.int32)
        wv   = f["IMG_WV"][0].astype(np.int32)
        ch0  = f["IMG_TIR1_TEMP"][:][tir1].astype(np.float32); del tir1
        ch1  = f["IMG_TIR2_TEMP"][:][tir2].astype(np.float32); del tir2
        ch2  = f["IMG_WV_TEMP"][:][wv].astype(np.float32);    del wv
        ch3  = f["IMG_MIR_TEMP"][:][mir].astype(np.float32);  del mir
        X    = f["X"][:]
        Y    = f["Y"][:]

    tir1_bt = ch0.copy()
    H, W    = ch0.shape
    bt_norm = np.empty((H, W, 4), dtype=np.float32)
    for c, ch in enumerate([ch0, ch1, ch2, ch3]):
        np.clip(ch, 180, 330, out=ch); ch -= 180; ch /= 150.0
        bt_norm[:, :, c] = ch
    del ch0, ch1, ch2, ch3
    gc.collect()
    return bt_norm, tir1_bt, X, Y


def sliding_window_predict(model, bt_norm, n_out):
    """
    Run model over bt_norm with sliding window.
    n_out : number of output channels (1 for rain, 4 for four-class)
    Returns (H, W, n_out) float16 — smallest possible representation.
    """
    H, W = bt_norm.shape[:2]
    acc  = np.zeros((H, W, n_out), dtype=np.float32)
    wmap = np.zeros((H, W),        dtype=np.float32)

    positions = [(i, j)
                 for i in range(0, H - PATCH_SIZE + 1, STRIDE)
                 for j in range(0, W - PATCH_SIZE + 1, STRIDE)]

    for bs in range(0, len(positions), BATCH_SIZE):
        pos   = positions[bs : bs + BATCH_SIZE]
        batch = np.stack([bt_norm[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
                          for i, j in pos], axis=0)

        # direct model call — leaner than model.predict() on CPU
        import tensorflow as tf
        with tf.device("/CPU:0"):
            preds = model(batch, training=False).numpy()   # (B,128,128,n_out)
        del batch

        for (i, j), pred in zip(pos, preds):
            acc[i:i+PATCH_SIZE, j:j+PATCH_SIZE, :]  += pred * _KERN[:,:,None]
            wmap[i:i+PATCH_SIZE, j:j+PATCH_SIZE]    += _KERN
        del preds
        gc.collect()

    acc /= (wmap[:,:,None] + 1e-8)
    del wmap
    return acc.astype(np.float16)   # float16 → half the disk/RAM footprint


# lon/lat cached as float32 — computed once
_lon_lat = None
def get_lonlat(X, Y):
    global _lon_lat
    if _lon_lat is None:
        xx, yy = np.meshgrid(X, Y)
        proj   = Proj(proj="geos", h=35786023, lon_0=82.5, sweep="x")
        tr     = Transformer.from_proj(proj, "epsg:4326", always_xy=True)
        lon, lat = tr.transform(xx, yy)
        _lon_lat = (lon.astype(np.float32), lat.astype(np.float32))
        del xx, yy, lon, lat; gc.collect()
        print("  lon/lat cached")
    return _lon_lat


# ============================================================
# 4. DISCOVER FILES
# ============================================================
pattern = os.path.join(INSAT_DIR, "3RIMG_*_L1C_SGP_V01R00.h5")
files   = sorted(glob.glob(pattern))
assert len(files) > 0, f"No files found in {INSAT_DIR}"

print(f"Found {len(files)} files")
first_ts   = parse_timestamp(files[0])
DATE_LABEL = first_ts.strftime("%d%b%Y").upper()
print(f"Date: {DATE_LABEL}")

# ============================================================
# 5. STAGE 1 — Rain model: run all files, save masks, DELETE model
# ============================================================
print("\n=== STAGE 1: Rain / No-Rain ===")
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)

from tensorflow.keras.models import load_model

rain_model = load_model(RAIN_MODEL_PATH, compile=False)
print("Rain model loaded")

tir1_cache = {}   # tir1_bt saved in RAM (small float16)
XY_ref     = None

for idx, fpath in enumerate(tqdm(files, desc="Stage 1")):
    bt_norm, tir1_bt, X, Y = read_hdf5(fpath)

    if XY_ref is None:
        XY_ref = (X, Y)   # save once for lon/lat grid

    # predict rain prob (channel 1 = rain)
    rain_out  = sliding_window_predict(rain_model, bt_norm, n_out=2)
    rain_mask = (rain_out[:,:,1] > RAIN_THRESHOLD)   # bool (H,W)
    del rain_out, bt_norm

    # save rain_mask + tir1_bt to disk as compressed npz
    np.savez_compressed(
        os.path.join(CACHE_DIR, f"stage1_{idx:03d}.npz"),
        rain_mask=rain_mask.astype(np.uint8),   # bool → uint8 saves space
        tir1_bt=tir1_bt.astype(np.float16)
    )
    del rain_mask, tir1_bt
    gc.collect()

# ── DELETE rain model completely ─────────────────────────────
del rain_model
gc.collect()
# Force TF to release its internal caches
import tensorflow as tf
tf.keras.backend.clear_session()
gc.collect()
print("Rain model deleted from memory")

# ============================================================
# 6. STAGE 2 — Four-class model: run all files, save probs, DELETE model
# ============================================================
print("\n=== STAGE 2: Four-class ===")
fc_model = load_model(FC_MODEL_PATH, compile=False)
print("Four-class model loaded")

for idx, fpath in enumerate(tqdm(files, desc="Stage 2")):
    bt_norm, _, _, _ = read_hdf5(fpath)   # re-read (bt_norm was freed)

    fc_out = sliding_window_predict(fc_model, bt_norm, n_out=4)
    # fc_out: (H,W,4) float16
    del bt_norm

    np.savez_compressed(
        os.path.join(CACHE_DIR, f"stage2_{idx:03d}.npz"),
        fc_probs=fc_out
    )
    del fc_out
    gc.collect()

# ── DELETE four-class model completely ───────────────────────
del fc_model
tf.keras.backend.clear_session()
gc.collect()
print("Four-class model deleted from memory")

# ============================================================
# 7. RENDER FRAMES — load cached results one file at a time
# ============================================================
print("\n=== RENDERING FRAMES ===")
lon, lat = get_lonlat(*XY_ref)
valid     = np.isfinite(lat) & np.isfinite(lon)
lon_clean = np.where(valid, lon, 0.0)
lat_clean = np.where(valid, lat, 0.0)

def _add_map(ax):
    ax.coastlines(resolution="10m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor="black")
    ax.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor="dimgray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.4,
                      color="gray", alpha=0.6, linestyle="--")
    gl.top_labels = False; gl.right_labels = False
    ax.set_extent(INDIA_EXTENT)

frames_A, frames_B = [], []

for idx, fpath in enumerate(tqdm(files, desc="Rendering")):
    ts = parse_timestamp(fpath)
    if ts is None:
        continue
    time_str = ts.strftime("%d %b %Y  %H:%M UTC")
    done     = "\u2588"*(idx+1) + "\u2591"*(len(files)-idx-1)

    # load cached stage results
    s1 = np.load(os.path.join(CACHE_DIR, f"stage1_{idx:03d}.npz"))
    s2 = np.load(os.path.join(CACHE_DIR, f"stage2_{idx:03d}.npz"))

    rain_mask = s1["rain_mask"].astype(bool)
    tir1_bt   = s1["tir1_bt"].astype(np.float32)
    fc_probs  = s2["fc_probs"].astype(np.float32)   # (H,W,4)
    del s1, s2

    # build category map
    pred_cls  = np.argmax(fc_probs, axis=-1).astype(np.float32)
    cat       = np.zeros(pred_cls.shape, dtype=np.float32)
    cat[valid & ~rain_mask]  = 1
    rainy = valid & rain_mask
    cat[rainy] = pred_cls[rainy] + 2
    del pred_cls, fc_probs

    tir1_plot = np.ma.array(tir1_bt,                      mask=~valid)
    rain_plot = np.ma.array(rain_mask.astype(np.float32), mask=~valid)
    cat_plot  = np.ma.array(cat,                           mask=~valid)
    del cat, rain_mask, tir1_bt

    # ── GIF-A: 3 panels ──────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(21, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    im = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                             cmap="RdYlBu_r", vmin=180, vmax=320,
                             transform=ccrs.PlateCarree(), shading="auto")
    _add_map(axes[0])
    plt.colorbar(im, ax=axes[0], pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    axes[1].pcolormesh(lon_clean, lat_clean, rain_plot,
                       cmap=RAIN_CMAP, vmin=0, vmax=1,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map(axes[1])
    axes[1].legend(handles=RAIN_LEGEND, loc="lower right", fontsize=8, framealpha=0.85)
    axes[1].set_title(f"Rain / No-Rain\n{time_str}", fontsize=10, fontweight="bold")

    axes[2].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=SUBA_CMAP, norm=norm_suba,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map(axes[2])
    axes[2].legend(handles=legend_suba, loc="lower right", fontsize=7, framealpha=0.85)
    axes[2].set_title(f"Rain Sub-classes\n{time_str}", fontsize=10, fontweight="bold")

    fig.suptitle(f"INSAT-3DR  |  GIF-A  [{idx+1:02d}/{len(files):02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)
    plt.tight_layout()
    pA = os.path.join(FRAMES_DIR, f"A_{idx:03d}.png")
    plt.savefig(pA, dpi=100, facecolor="white")
    plt.close(fig); frames_A.append(pA)

    # ── GIF-B: 2 panels ──────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    im = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                             cmap="RdYlBu_r", vmin=180, vmax=320,
                             transform=ccrs.PlateCarree(), shading="auto")
    _add_map(axes[0])
    plt.colorbar(im, ax=axes[0], pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    axes[1].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=SUBA_CMAP, norm=norm_suba,
                       transform=ccrs.PlateCarree(), shading="auto")
    _add_map(axes[1])
    axes[1].legend(handles=legend_suba, loc="lower right", fontsize=7, framealpha=0.85)
    axes[1].set_title(f"Four-class + No Rain\n{time_str}", fontsize=10, fontweight="bold")

    fig.suptitle(f"INSAT-3DR  |  GIF-B  [{idx+1:02d}/{len(files):02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)
    plt.tight_layout()
    pB = os.path.join(FRAMES_DIR, f"B_{idx:03d}.png")
    plt.savefig(pB, dpi=100, facecolor="white")
    plt.close(fig); frames_B.append(pB)

    del tir1_plot, rain_plot, cat_plot, fig, axes
    gc.collect()

print(f"{len(frames_A)} frames rendered")

# ============================================================
# 8. COMPILE GIFS  (streaming — never holds all frames in RAM)
# ============================================================
def compile_gif(frame_list, out_hq, out_std):
    n        = len(frame_list)
    ms       = [int(1000/FPS)] * n
    ms[0]    = 2500
    ms[-1]   = 4000

    def _pil(p, sz=None):
        img = Image.open(p).convert("RGBA")
        if sz and img.size != sz: img = img.resize(sz, Image.LANCZOS)
        return img

    first = _pil(frame_list[0]); sz = first.size
    first.save(out_hq, save_all=True,
               append_images=(_pil(p, sz) for p in frame_list[1:]),
               duration=ms, loop=0, optimize=True)
    del first; gc.collect()
    print(f"  HQ  -> {out_hq}")

    tw, th = sz
    with imageio.get_writer(out_std, mode="I", loop=0) as w:
        for p in frame_list:
            fr = imageio.imread(p)
            if fr.shape[:2] != (th, tw):
                fr = np.array(Image.fromarray(fr).resize((tw, th), Image.LANCZOS))
            w.append_data(fr); del fr
    gc.collect()
    print(f"  STD -> {out_std}")

print("\n=== COMPILING GIFs ===")
compile_gif(frames_A,
            os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}_hq.gif"),
            os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}.gif"))

compile_gif(frames_B,
            os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}_hq.gif"),
            os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}.gif"))

# ============================================================
# 9. DISPLAY
# ============================================================
print("\n=== GIF-A: TIR1 | Rain/No-Rain | Sub-classes ===")
display(IPImage(filename=os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}.gif")))
print("\n=== GIF-B: TIR1 | Four-class + No-Rain ===")
display(IPImage(filename=os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}.gif")))

In [ ]:
"""
INSAT Two-Stage GIF Generator — Minimal RAM Edition (FINAL FIX)

Root cause found from HDF5 Projection_Information attributes:
  ✓ grid_mapping_name = 'mercator'  (NOT geos — all previous fixes were wrong projection)
  ✓ longitude_of_projection_origin = 75.0
  ✓ standard_parallel = 0.0
  ✓ semi_major_axis = 6378137.0  (WGS84)
  ✓ semi_minor_axis = 6356752.3142
  ✓ X, Y already in metres (confirmed by units='m' attribute)
  ✓ upper_left_xy = [-6122572, +6413525] matches X.min(), Y.max()
  ✓ Grid extent: lon 20°–130°E, lat -50°–50°N (from corner lat/lon attributes)

Other fixes retained:
  ✓ shading="nearest" on all pcolormesh calls
  ✓ Orientation guard (flipud if needed)
"""

# ============================================================
# 0. IMPORTS + FORCE CPU BEFORE TF LOADS
# ============================================================
import os, gc
os.environ["CUDA_VISIBLE_DEVICES"]   = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"]   = "3"
os.environ["TF_XLA_FLAGS"]           = "--tf_xla_enable_xla_devices=false"
os.environ["TF_ENABLE_ONEDNN_OPTS"]  = "0"
os.environ["OMP_NUM_THREADS"]        = "4"
os.environ["TF_NUM_INTRAOP_THREADS"] = "4"
os.environ["TF_NUM_INTEROP_THREADS"] = "2"

import warnings
warnings.filterwarnings("ignore")

import re, glob
from datetime import datetime

import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from pyproj import Proj, Transformer
from PIL import Image
from IPython.display import Image as IPImage, display
import imageio.v2 as imageio
from tqdm import tqdm

# ============================================================
# 1. CONFIG
# ============================================================
RAIN_MODEL_PATH = "/home/shruti/Prep_project/notebooks/models/4_inputs_BT_only/rain_norain_BT_New.keras"
FC_MODEL_PATH   = "/home/shruti/Prep_project/notebooks/models/4_inputs_BT_only/fourclasses_layernorm_BT_only.keras"

INSAT_DIR    = "/home/shruti/Documents/insat_data"
OUTPUT_DIR   = "/home/shruti/Downloads/insat_gif_strat_conv222222"
FRAMES_DIR   = "/tmp/insat_frames/"
CACHE_DIR    = "/tmp/insat_cache/"

INDIA_EXTENT = [68, 97, 7, 37]
PATCH_SIZE   = 128
STRIDE       = 68
FPS          = 2
BATCH_SIZE   = 4

RAIN_THRESHOLD = 0.50

for d in [OUTPUT_DIR, FRAMES_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# 2. COLORMAPS
# ============================================================
SUBA_CMAP   = ListedColormap(["#CCCCCC", "#FFFFFF", "#3A86FF",
                               "#D32F2F", "#FF7043", "#FFC107"])
bounds_suba = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm_suba   = BoundaryNorm(bounds_suba, SUBA_CMAP.N)

legend_suba = [
    mpatches.Patch(color="#3A86FF", label="Stratiform"),
    mpatches.Patch(color="#D32F2F", label="Deep Convective"),
    mpatches.Patch(color="#FF7043", label="Shallow Conv (isolated)"),
    mpatches.Patch(color="#FFC107", label="Shallow Conv (non-iso)"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
    mpatches.Patch(color="#CCCCCC", label="No Data"),
]
RAIN_CMAP   = ListedColormap(["#FFFFFF", "#2196F3"])
RAIN_LEGEND = [
    mpatches.Patch(color="#2196F3", label="Rain"),
    mpatches.Patch(color="#FFFFFF", label="No Rain", edgecolor="#AAAAAA"),
]

# ============================================================
# 3. HELPERS
# ============================================================
def parse_timestamp(fpath):
    m = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{4})', os.path.basename(fpath))
    if m:
        return datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")
    return None


def _make_gaussian_kernel(size):
    rng    = np.linspace(-1, 1, size, dtype=np.float32)
    g1d    = np.exp(-0.5 * (rng / 0.35) ** 2)
    kernel = np.outer(g1d, g1d).astype(np.float32)
    kernel /= kernel.sum()
    return kernel

_KERN = _make_gaussian_kernel(PATCH_SIZE)


def read_hdf5(fpath):
    """Read one HDF5 file, return bt_norm (H,W,4) float32, tir1_bt, X, Y."""
    with h5py.File(fpath, "r") as f:
        tir1 = f["IMG_TIR1"][0].astype(np.int32)
        tir2 = f["IMG_TIR2"][0].astype(np.int32)
        mir  = f["IMG_MIR"][0].astype(np.int32)
        wv   = f["IMG_WV"][0].astype(np.int32)
        ch0  = f["IMG_TIR1_TEMP"][:][tir1].astype(np.float32); del tir1
        ch1  = f["IMG_TIR2_TEMP"][:][tir2].astype(np.float32); del tir2
        ch2  = f["IMG_WV_TEMP"][:][wv].astype(np.float32);     del wv
        ch3  = f["IMG_MIR_TEMP"][:][mir].astype(np.float32);   del mir
        X    = f["X"][:].astype(np.float64)   # metres (confirmed by units='m')
        Y    = f["Y"][:].astype(np.float64)   # metres

    tir1_bt = ch0.copy()
    H, W    = ch0.shape
    bt_norm = np.empty((H, W, 4), dtype=np.float32)
    for c, ch in enumerate([ch0, ch1, ch2, ch3]):
        np.clip(ch, 180, 330, out=ch); ch -= 180; ch /= 150.0
        bt_norm[:, :, c] = ch
    del ch0, ch1, ch2, ch3
    gc.collect()
    return bt_norm, tir1_bt, X, Y


def sliding_window_predict(model, bt_norm, n_out):
    H, W = bt_norm.shape[:2]

    # Pad so every pixel gets covered by at least one patch
    pad_h = (PATCH_SIZE - H % PATCH_SIZE) % PATCH_SIZE if H % STRIDE != 0 else 0
    pad_w = (PATCH_SIZE - W % PATCH_SIZE) % PATCH_SIZE if W % STRIDE != 0 else 0

    # Reflect-pad to avoid introducing artificial edges
    if pad_h > 0 or pad_w > 0:
        bt_pad = np.pad(bt_norm,
                        ((0, pad_h), (0, pad_w), (0, 0)),
                        mode="reflect")
    else:
        bt_pad = bt_norm

    Hp, Wp = bt_pad.shape[:2]
    acc  = np.zeros((Hp, Wp, n_out), dtype=np.float32)
    wmap = np.zeros((Hp, Wp),        dtype=np.float32)

    positions = [(i, j)
                 for i in range(0, Hp - PATCH_SIZE + 1, STRIDE)
                 for j in range(0, Wp - PATCH_SIZE + 1, STRIDE)]

    for bs in range(0, len(positions), BATCH_SIZE):
        pos   = positions[bs : bs + BATCH_SIZE]
        batch = np.stack([bt_pad[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
                          for i, j in pos], axis=0)

        import tensorflow as tf
        with tf.device("/CPU:0"):
            preds = model(batch, training=False).numpy()
        del batch

        for (i, j), pred in zip(pos, preds):
            acc[i:i+PATCH_SIZE, j:j+PATCH_SIZE, :]  += pred * _KERN[:,:,None]
            wmap[i:i+PATCH_SIZE, j:j+PATCH_SIZE]    += _KERN
        del preds
        gc.collect()

    acc /= (wmap[:,:,None] + 1e-8)
    del wmap, bt_pad

    # Crop back to original size
    return acc[:H, :W].astype(np.float16)


# ============================================================
# FIXED get_lonlat — uses MERCATOR projection (from file attributes)
#
# Projection_Information attributes confirm:
#   grid_mapping_name             = 'mercator'
#   longitude_of_projection_origin = 75.0
#   standard_parallel             = 0.0
#   semi_major_axis               = 6378137.0   (WGS84)
#   semi_minor_axis               = 6356752.3142
#   X, Y units                   = metres
#   upper_left_xy(meters)        = [-6122572, +6413525]  ← matches X.min(), Y.max()
#   Grid corners (degrees):
#     upper_left  = [50°N, 20°E]
#     upper_right = [50°N, 130°E]
#     lower_left  = [50°S, 20°E]
#     lower_right = [50°S, 130°E]
# ============================================================
_lon_lat = None

def get_lonlat(X, Y):
    global _lon_lat
    if _lon_lat is None:
        # Build 2D coordinate grids
        # X → columns (easting),  Y → rows (northing)
        xx, yy = np.meshgrid(X, Y)   # xx.shape = yy.shape = (nrows, ncols)

        # ── CORRECT projection: Mercator ─────────────────────────────────────
        # Parameters read directly from Projection_Information dataset attrs:
        #   longitude_of_projection_origin = 75.0
        #   standard_parallel              = 0.0
        #   semi_major_axis                = 6378137.0
        #   semi_minor_axis                = 6356752.3142
        proj_merc = Proj(
            proj   = "merc",
            lon_0  = 75.0,           # longitude_of_projection_origin
            lat_ts = 0.0,            # standard_parallel
            a      = 6378137.0,      # semi_major_axis (WGS84)
            b      = 6356752.3142,   # semi_minor_axis (WGS84)
            units  = "m"
        )
        target_crs = "epsg:4326"     # WGS84 geographic (lon, lat degrees)

        tr = Transformer.from_proj(proj_merc, target_crs, always_xy=True)
        # always_xy=True: input order = (easting=xx, northing=yy)
        #                 output order = (longitude, latitude)
        lon, lat = tr.transform(xx, yy)
        del xx, yy

        # ── Orientation guard ─────────────────────────────────────────────────
        # In Mercator, Y increases northward, but array row-0 = Y.max (north)
        # because upper_left_xy has Y = +6413525 (max).
        # So row-0 should already be the northernmost row — verify and flip if not.
        row0_lat  = np.nanmean(lat[0,  :])
        rowN_lat  = np.nanmean(lat[-1, :])
        if row0_lat < rowN_lat:
            lat = np.flipud(lat)
            lon = np.flipud(lon)
            print("  [fix] lat array flipped north-up")
        else:
            print("  [ok]  lat array already north-up")

        _lon_lat = (lon.astype(np.float32), lat.astype(np.float32))
        del lon, lat
        gc.collect()

        print(f"  lon/lat cached: "
              f"lat [{_lon_lat[1].min():.2f} – {_lon_lat[1].max():.2f}]  "
              f"lon [{_lon_lat[0].min():.2f} – {_lon_lat[0].max():.2f}]")
        print(f"  Expected:       lat [-50 – +50]  lon [20 – 130]")
    return _lon_lat


# ============================================================
# 4. DISCOVER FILES
# ============================================================
pattern = os.path.join(INSAT_DIR, "3RIMG_*_L1C_SGP_V01R00.h5")
files   = sorted(glob.glob(pattern))
assert len(files) > 0, f"No files found in {INSAT_DIR}"

print(f"Found {len(files)} files")
first_ts   = parse_timestamp(files[0])
DATE_LABEL = first_ts.strftime("%d%b%Y").upper()
print(f"Date: {DATE_LABEL}")

# ============================================================
# 5. STAGE 1 — Rain model
# ============================================================
print("\n=== STAGE 1: Rain / No-Rain ===")
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)

from tensorflow.keras.models import load_model

rain_model = load_model(RAIN_MODEL_PATH, compile=False)
print("Rain model loaded")

XY_ref = None

for idx, fpath in enumerate(tqdm(files, desc="Stage 1")):
    bt_norm, tir1_bt, X, Y = read_hdf5(fpath)

    if XY_ref is None:
        XY_ref = (X, Y)

    rain_out  = sliding_window_predict(rain_model, bt_norm, n_out=2)
    rain_mask = (rain_out[:,:,1] > RAIN_THRESHOLD)
    del rain_out, bt_norm

    np.savez_compressed(
        os.path.join(CACHE_DIR, f"stage1_{idx:03d}.npz"),
        rain_mask=rain_mask.astype(np.uint8),
        tir1_bt=tir1_bt.astype(np.float16)
    )
    del rain_mask, tir1_bt
    gc.collect()

del rain_model
gc.collect()
tf.keras.backend.clear_session()
gc.collect()
print("Rain model deleted from memory")

# ============================================================
# 6. STAGE 2 — Four-class model
# ============================================================
print("\n=== STAGE 2: Four-class ===")
fc_model = load_model(FC_MODEL_PATH, compile=False)
print("Four-class model loaded")

for idx, fpath in enumerate(tqdm(files, desc="Stage 2")):
    bt_norm, _, _, _ = read_hdf5(fpath)

    fc_out = sliding_window_predict(fc_model, bt_norm, n_out=4)
    del bt_norm

    np.savez_compressed(
        os.path.join(CACHE_DIR, f"stage2_{idx:03d}.npz"),
        fc_probs=fc_out
    )
    del fc_out
    gc.collect()

del fc_model
tf.keras.backend.clear_session()
gc.collect()
print("Four-class model deleted from memory")

# ============================================================
# 7. RENDER FRAMES
# ============================================================
print("\n=== RENDERING FRAMES ===")
lon, lat = get_lonlat(*XY_ref)
valid     = np.isfinite(lat) & np.isfinite(lon)
lon_clean = np.where(valid, lon, 0.0)
lat_clean = np.where(valid, lat, 0.0)


def _add_map(ax):
    ax.coastlines(resolution="10m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.7, edgecolor="black")
    ax.add_feature(cfeature.STATES,  linewidth=0.4, edgecolor="dimgray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.4,
                      color="gray", alpha=0.6, linestyle="--")
    gl.top_labels = False; gl.right_labels = False
    ax.set_extent(INDIA_EXTENT)


frames_A, frames_B = [], []

for idx, fpath in enumerate(tqdm(files, desc="Rendering")):
    ts = parse_timestamp(fpath)
    if ts is None:
        continue
    time_str = ts.strftime("%d %b %Y  %H:%M UTC")
    done     = "\u2588"*(idx+1) + "\u2591"*(len(files)-idx-1)

    s1 = np.load(os.path.join(CACHE_DIR, f"stage1_{idx:03d}.npz"))
    s2 = np.load(os.path.join(CACHE_DIR, f"stage2_{idx:03d}.npz"))

    rain_mask = s1["rain_mask"].astype(bool)
    tir1_bt   = s1["tir1_bt"].astype(np.float32)
    fc_probs  = s2["fc_probs"].astype(np.float32)
    del s1, s2

    pred_cls  = np.argmax(fc_probs, axis=-1).astype(np.float32)
    cat       = np.zeros(pred_cls.shape, dtype=np.float32)
    cat[valid & ~rain_mask] = 1
    rainy = valid & rain_mask
    cat[rainy] = pred_cls[rainy] + 2
    del pred_cls, fc_probs

    tir1_plot = np.ma.array(tir1_bt,                      mask=~valid)
    rain_plot = np.ma.array(rain_mask.astype(np.float32), mask=~valid)
    cat_plot  = np.ma.array(cat,                           mask=~valid)
    del cat, rain_mask, tir1_bt

    # ── GIF-A: 3 panels ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(21, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    im = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                             cmap="RdYlBu_r", vmin=180, vmax=320,
                             transform=ccrs.PlateCarree(),
                             shading="nearest")
    _add_map(axes[0])
    plt.colorbar(im, ax=axes[0], pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    axes[1].pcolormesh(lon_clean, lat_clean, rain_plot,
                       cmap=RAIN_CMAP, vmin=0, vmax=1,
                       transform=ccrs.PlateCarree(),
                       shading="nearest")
    _add_map(axes[1])
    axes[1].legend(handles=RAIN_LEGEND, loc="lower right", fontsize=8, framealpha=0.85)
    axes[1].set_title(f"Rain / No-Rain\n{time_str}", fontsize=10, fontweight="bold")

    axes[2].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=SUBA_CMAP, norm=norm_suba,
                       transform=ccrs.PlateCarree(),
                       shading="nearest")
    _add_map(axes[2])
    axes[2].legend(handles=legend_suba, loc="lower right", fontsize=7, framealpha=0.85)
    axes[2].set_title(f"Rain Sub-classes\n{time_str}", fontsize=10, fontweight="bold")

    fig.suptitle(f"INSAT-3DR  |  GIF-A  [{idx+1:02d}/{len(files):02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)
    plt.tight_layout()
    pA = os.path.join(FRAMES_DIR, f"A_{idx:03d}.png")
    plt.savefig(pA, dpi=100, facecolor="white")
    plt.close(fig); frames_A.append(pA)

    # ── GIF-B: 2 panels ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 7),
                             subplot_kw={"projection": ccrs.PlateCarree()})
    fig.patch.set_facecolor("white")

    im = axes[0].pcolormesh(lon_clean, lat_clean, tir1_plot,
                             cmap="RdYlBu_r", vmin=180, vmax=320,
                             transform=ccrs.PlateCarree(),
                             shading="nearest")
    _add_map(axes[0])
    plt.colorbar(im, ax=axes[0], pad=0.02, shrink=0.92).set_label("BT (K)", fontsize=9)
    axes[0].set_title(f"TIR1 Brightness Temp\n{time_str}", fontsize=10, fontweight="bold")

    axes[1].pcolormesh(lon_clean, lat_clean, cat_plot,
                       cmap=SUBA_CMAP, norm=norm_suba,
                       transform=ccrs.PlateCarree(),
                       shading="nearest")
    _add_map(axes[1])
    axes[1].legend(handles=legend_suba, loc="lower right", fontsize=7, framealpha=0.85)
    axes[1].set_title(f"Four-class + No Rain\n{time_str}", fontsize=10, fontweight="bold")

    fig.suptitle(f"INSAT-3DR  |  GIF-B  [{idx+1:02d}/{len(files):02d}]  {done}",
                 fontsize=12, fontweight="bold", y=1.005)
    plt.tight_layout()
    pB = os.path.join(FRAMES_DIR, f"B_{idx:03d}.png")
    plt.savefig(pB, dpi=100, facecolor="white")
    plt.close(fig); frames_B.append(pB)

    del tir1_plot, rain_plot, cat_plot, fig, axes
    gc.collect()

print(f"{len(frames_A)} frames rendered")

# ============================================================
# 8. COMPILE GIFS
# ============================================================
def compile_gif(frame_list, out_hq, out_std):
    n     = len(frame_list)
    ms    = [int(1000/FPS)] * n
    ms[0] = 2500
    ms[-1]= 4000

    def _pil(p, sz=None):
        img = Image.open(p).convert("RGBA")
        if sz and img.size != sz: img = img.resize(sz, Image.LANCZOS)
        return img

    first = _pil(frame_list[0]); sz = first.size
    first.save(out_hq, save_all=True,
               append_images=(_pil(p, sz) for p in frame_list[1:]),
               duration=ms, loop=0, optimize=True)
    del first; gc.collect()
    print(f"  HQ  -> {out_hq}")

    tw, th = sz
    with imageio.get_writer(out_std, mode="I", loop=0) as w:
        for p in frame_list:
            fr = imageio.imread(p)
            if fr.shape[:2] != (th, tw):
                fr = np.array(Image.fromarray(fr).resize((tw, th), Image.LANCZOS))
            w.append_data(fr); del fr
    gc.collect()
    print(f"  STD -> {out_std}")

print("\n=== COMPILING GIFs ===")
compile_gif(frames_A,
            os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}_hq.gif"),
            os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}.gif"))

compile_gif(frames_B,
            os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}_hq.gif"),
            os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}.gif"))

# ============================================================
# 9. DISPLAY
# ============================================================
print("\n=== GIF-A: TIR1 | Rain/No-Rain | Sub-classes ===")
display(IPImage(filename=os.path.join(OUTPUT_DIR, f"A_tir1_rain_4class_{DATE_LABEL}.gif")))
print("\n=== GIF-B: TIR1 | Four-class + No-Rain ===")
display(IPImage(filename=os.path.join(OUTPUT_DIR, f"B_tir1_4class_{DATE_LABEL}.gif")))

In [ ]:
import h5py, numpy as np
from pyproj import Proj, Transformer

fpath = "/home/shruti/Documents/insat_data/3RIMG_20APR2026_1015_L1C_SGP_V01R00.h5"

with h5py.File(fpath, "r") as f:
    X = f["X"][:].astype(np.float64)
    Y = f["Y"][:].astype(np.float64)

print(f"X: min={X.min():.1f}, max={X.max():.1f}")
print(f"Y: min={Y.min():.1f}, max={Y.max():.1f}")

# Apply unit fix
if np.abs(X).max() < 1e5:
    X *= 1000.0; Y *= 1000.0
    print("Units were km → converted to metres")

xx, yy = np.meshgrid(X, Y)
proj = Proj(proj="geos", h=35785831, lon_0=82.0, sweep="y", units="m")
tr   = Transformer.from_proj(proj, "epsg:4326", always_xy=True)
lon, lat = tr.transform(xx, yy)

print(f"Lat range: {np.nanmin(lat):.2f} to {np.nanmax(lat):.2f}")
print(f"Lon range: {np.nanmin(lon):.2f} to {np.nanmax(lon):.2f}")
# Expected: lat ~-10 to +45, lon ~50 to +115

In [ ]:
import h5py

fpath = "/home/shruti/Documents/insat_data/3RIMG_20APR2026_1015_L1C_SGP_V01R00.h5"

with h5py.File(fpath, "r") as f:
    # Print ALL root attributes
    print("=== ROOT ATTRIBUTES ===")
    for k, v in f.attrs.items():
        print(f"  {k}: {v}")
    
    # Print dataset names + their attributes
    print("\n=== DATASETS ===")
    def show(name, obj):
        print(f"  {name}: shape={getattr(obj,'shape','—')}")
        for k, v in obj.attrs.items():
            print(f"      .{k} = {v}")
    f.visititems(show)